# Distran: adaptive personalization with CNN and GPT-style content encoders (05)

Run all cells on a CPU runtime via the installed Colab CLI. This notebook executes the Phase C proposal from
[ADAPTIVE_PERSONALIZATION_RESEARCH.md](ADAPTIVE_PERSONALIZATION_RESEARCH.md) and records measured results in
[ADAPTIVE_PERSONALIZATION_RESULTS.md](ADAPTIVE_PERSONALIZATION_RESULTS.md).

**Name and classification of model.** Name: **Distran APC-1** (`distran-apc-v1`), an adaptive preference classifier:
a shared logistic prior `theta` plus a private per-user residual `delta_u` in
`p(u,i,c) = sigmoid(b + b_u + (theta + delta_u) . x(u,i,c))`, trained with online SGD under chronological
predict-before-learn replay on the sampled Amazon 5-core interactions. It is a feature-based linear classifier with
per-user adaptation (feature/score fusion), not parameter averaging across artifacts, and not a collaborative model.
Two neural content encoders are trained on the pinned Kaggle metadata corpus alongside it: a **TextCNN**
(1D convolutions over token embeddings, max-over-time pooling, score-bucket head) and a **mini-GPT**
(decoder-only causal transformer trained with next-token prediction), both implemented in pure NumPy, plus the
existing TF-IDF/SVD encoder as baseline. Serving is local JSON coefficients; nothing is promoted to the app.

Data: complete Amazon 5-core rating-only files (deterministic 1/128 user sample, salted pseudonyms, same rule as
notebook 03) and the pinned TMDB movies + HowLongToBeat snapshots (same handles/versions as notebook 04). No app
user data, no provider credentials beyond public Kaggle downloads. Raw IDs stay in memory; exports use salted hashes.


In [ ]:
# @title Pin dependencies
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "numpy==2.2.6", "pandas==2.2.3", "scikit-learn==1.6.1",
                       "kagglehub[pandas-datasets]==0.4.3"])


0

In [ ]:
# @title Install the exact experiment source and app snapshots
from pathlib import Path
import hashlib, json, sys
root = Path("/content/distran-adaptive")
root.mkdir(parents=True, exist_ok=True)
files = {
    "adaptive_experiment.py": "\"\"\"Adaptive personalization experiment (notebook 05); run on CPU in Colab.\n\nTrains two neural content encoders on the pinned Kaggle metadata corpus -- a\nTextCNN score-bucket classifier (1D convolutions over token embeddings) and a\nsmall decoder-only GPT-style language model (causal transformer, next-token\npretraining) -- then trains the adaptive preference classifier proposed in\nnotebooks/ADAPTIVE_PERSONALIZATION_RESEARCH.md on the sampled Amazon 5-core\ninteractions: a shared logistic prior plus a private per-user residual updated\nonline with chronological predict-before-learn replay.\n\nRaw source IDs are used only in memory; exported coefficients are keyed by\nsalted hashes. Research artifacts only; nothing here is promoted to the app.\n\"\"\"\nimport csv, gzip, hashlib, json, math, os, platform, re, subprocess, sys, time, unicodedata, urllib.request\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\nSEED = 42\nAMAZON_BASE = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/5core/rating_only/'\nKAGGLE = {'movies': 'balaka18/tmdb-top-10000-popular-movies-dataset/versions/2',\n          'hltb': 'zaireali/howlongtobeat-games-scraper-2162025/versions/1'}\nKAGGLE_FILES = {'movies': 'movies_tmdb_popular.csv', 'hltb': 'hltb_data.csv'}\nHLTB_SAMPLE = 10000\nMAX_TOKENS = 48\nMODEL_NAME = 'Distran APC-1 (Adaptive Preference Classifier: shared-prior online logistic with per-user residual)'\nMODEL_ID = 'distran-apc-v1'\nMODULE_VERSION = 3\nFEATURES = ['item_pop_pct', 'item_quality', 'item_dom_game', 'item_dom_screen', 'item_recency_90d',\n            'user_activity', 'user_pos_rate', 'user_aff_game', 'user_aff_screen', 'user_recency_30d',\n            'inter_aff_game', 'inter_aff_screen', 'collab_pct']\n\n\ndef clean(value):\n    if value is None or (isinstance(value, float) and not np.isfinite(value)):\n        return ''\n    return re.sub(r'\\s+', ' ', str(value)).strip()\n\n\ndef title_key(value):\n    value = unicodedata.normalize('NFKD', clean(value)).encode('ascii', 'ignore').decode().lower()\n    value = value.replace(\"'\", '')\n    return re.sub(r'[^a-z0-9]+', ' ', value).strip()\n\n\ndef split_bucket(title):\n    return int(hashlib.sha256(title_key(title).encode()).hexdigest()[:8], 16) % 10\n\n\ndef text_key(text):\n    return re.sub(r'\\s+', ' ', re.sub(r\"[^a-z0-9'\\s]\", ' ', clean(text).lower())).strip()\n\n\ndef sigmoid(z):\n    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))\n\n\nclass Adam:\n    def __init__(self, params, lr=0.01, b1=0.9, b2=0.999, eps=1e-8):\n        self.params = params\n        self.lr, self.b1, self.b2, self.eps = lr, b1, b2, eps\n        self.m = [np.zeros_like(p) for p in params]\n        self.v = [np.zeros_like(p) for p in params]\n        self.t = 0\n\n    def step(self, grads):\n        self.t += 1\n        for p, g, m, v in zip(self.params, grads, self.m, self.v):\n            m *= self.b1; m += (1 - self.b1) * g\n            v *= self.b2; v += (1 - self.b2) * g * g\n            mh = m / (1 - self.b1 ** self.t); vh = v / (1 - self.b2 ** self.t)\n            p -= self.lr * mh / (np.sqrt(vh) + self.eps)\n\n\n# ---------------------------------------------------------------- data\n\ndef prepare_amazon(root):\n    salt = os.urandom(32)\n    events = []; sources = []; user_sets = []\n    for domain, category in [('game', 'Video_Games'), ('screen', 'Movies_and_TV')]:\n        path = root / (category + '.csv.gz'); url = AMAZON_BASE + path.name\n        print('Downloading complete file:', category, flush=True)\n        if not path.exists():\n            with urllib.request.urlopen(url, timeout=180) as response, path.open('wb') as out:\n                while chunk := response.read(1024 * 1024):\n                    out.write(chunk)\n        digest = hashlib.sha256(path.read_bytes()).hexdigest()\n        users = set(); count = 0; kept = 0\n        with gzip.open(path, 'rt') as stream:\n            for row in csv.DictReader(stream):\n                count += 1; raw = row['user_id']; users.add(raw)\n                if int.from_bytes(hashlib.sha256(raw.encode()).digest()[:8], 'big') % 128:\n                    continue\n                rating = float(row['rating']); timestamp = int(row['timestamp'])\n                if not np.isfinite(rating) or not 1 <= rating <= 5:\n                    raise ValueError('Invalid rating')\n                user = hashlib.sha256(salt + raw.encode()).hexdigest()\n                events.append(dict(user_id=user, item_id=domain + ':' + row['parent_asin'],\n                                   rating=rating, timestamp=timestamp)); kept += 1\n        user_sets.append(users)\n        sources.append(dict(url=url, sha256=digest, bytes=path.stat().st_size, rows=count,\n                            users=len(users), sampled_rows=kept))\n        print(category, sources[-1], flush=True)\n    overlap = len(user_sets[0] & user_sets[1]); del user_sets\n    unique = {(e['user_id'], e['item_id'], e['timestamp']): e for e in events}\n    audit = dict(sources=sources, full_file_overlap_users=overlap, duplicates_removed=len(events) - len(unique),\n                 sample='SHA256(source_user_id) first 8 bytes big endian modulo 128 == 0; complete file scans; no future-based user filtering',\n                 pseudonyms='Random per-run salt, not exported; sampled cohort deterministic, pseudonyms differ each run',\n                 limitations='Publisher 5-core selection uses complete history; screen combines movies and TV. Same sampling rule as notebook 03 so cohorts are comparable.')\n    return list(unique.values()), audit\n\n\ndef prepare_corpus(root):\n    import kagglehub\n    import pandas as pd\n    raw = {}; sources = {}\n    for key, handle in KAGGLE.items():\n        directory = Path(kagglehub.dataset_download(handle)); path = directory / KAGGLE_FILES[key]\n        raw[key] = pd.read_csv(path, low_memory=False)\n        sources[key] = dict(handle=handle, file=path.name, sha256=hashlib.sha256(path.read_bytes()).hexdigest(),\n                            bytes=path.stat().st_size, rows=len(raw[key]))\n        print('Kaggle', key, sources[key], flush=True)\n    docs = []\n    for _, row in raw['movies'].iterrows():\n        title = clean(row.get('title'))\n        if not title:\n            continue\n        votes = float(pd.to_numeric(pd.Series([row.get('vote_count')]), errors='coerce').iloc[0])\n        score = float(pd.to_numeric(pd.Series([row.get('vote_average')]), errors='coerce').iloc[0])\n        score = score if np.isfinite(score) and np.isfinite(votes) and votes >= 5 and 0 <= score <= 10 else None\n        docs.append(dict(title=title, source='tmdb', score=score,\n                         text=title + '. ' + clean(row.get('overview')) + ' language_' + clean(row.get('original_lang'))))\n    hltb = raw['hltb']\n    hltb = hltb[hltb['game_game_name'].notna()].copy()\n    hltb = hltb.sort_values(['game_game_id', 'platform_platform'], kind='stable').drop_duplicates('game_game_id')\n    hltb['order'] = hltb['game_game_name'].map(lambda t: hashlib.sha256(title_key(t).encode()).hexdigest())\n    hltb = hltb.sort_values('order', kind='stable').head(HLTB_SAMPLE)\n    for _, row in hltb.iterrows():\n        title = clean(row.get('game_game_name'))\n        reviews = float(pd.to_numeric(pd.Series([row.get('game_count_review')]), errors='coerce').iloc[0])\n        score = float(pd.to_numeric(pd.Series([row.get('game_review_score')]), errors='coerce').iloc[0])\n        score = score / 10.0 if np.isfinite(score) and np.isfinite(reviews) and reviews >= 5 and 0 < score <= 100 else None\n        doc = ' '.join(clean(row.get(c)) for c in ('game_profile_summary', 'game_profile_genre', 'game_profile_dev', 'game_profile_pub'))\n        docs.append(dict(title=title, source='hltb', score=score, text=title + '. ' + doc))\n    for d in docs:\n        d['bucket'] = split_bucket(d['title'])\n        d['label'] = None if d['score'] is None else (0 if d['score'] < 6 else 1 if d['score'] <= 7.5 else 2)\n    audit = dict(sources=sources, docs=len(docs), scored=sum(d['label'] is not None for d in docs),\n                 hltb_sample=HLTB_SAMPLE, hltb_policy='deterministic SHA256-title-ordered subset; one row per game_game_id',\n                 split='SHA256(normalized title)[:8] hex mod 10; buckets 0-5 train, 6-7 validation, 8-9 test',\n                 score_policy='TMDB vote_average (0-10) with vote_count>=5; HLTB game_review_score/10 with game_count_review>=5; buckets low<6, mid 6-7.5, high>7.5')\n    return docs, audit\n\n\ndef synthetic_events(rng, n_users=300, n_items=400, days=60):\n    items = [(('game' if i % 2 else 'screen') + ':item%03d' % i, float(rng.random())) for i in range(n_items)]\n    events = []\n    base = 1704067200000  # 2024-01-01 in ms\n    for u in range(n_users):\n        user = 'synthetic:%03d' % u\n        pref_game = float(rng.random())\n        for it, quality in items:\n            if rng.random() > 0.08:\n                continue\n            dom = it.split(':')[0]\n            affinity = pref_game if dom == 'game' else 1 - pref_game\n            p = 0.15 + 0.6 * affinity * quality + 0.2 * quality\n            rating = 5 if rng.random() < p else (1 if rng.random() < 0.5 else 3)\n            day = int(rng.integers(0, days))\n            events.append(dict(user_id=user, item_id=it, rating=float(rating),\n                               timestamp=base + day * 86400000 + int(rng.integers(0, 86400000))))\n    return events, dict(note='synthetic local test', users=n_users, items=n_items)\n\n\ndef synthetic_corpus(rng, n=1200):\n    game_words = 'space shooter loot online boss raid squad competitive arena'.split()\n    movie_words = 'family drama romance wedding love story musical comedy'.split()\n    docs = []\n    for i in range(n):\n        words = game_words if i % 2 else movie_words\n        k = int(rng.integers(10, 30))\n        toks = rng.choice(words, size=k).tolist()\n        text = ' '.join(toks) + ' title%04d' % i\n        frac = float(np.mean([w in game_words for w in toks]))\n        score = 6.0 + (3.0 if i % 2 else -3.0) * (frac if i % 2 else 1 - frac) * 0 + (3.0 if i % 2 else -2.0) + float(rng.normal(0, .5))\n        score = float(np.clip(score, 1, 10))\n        docs.append(dict(title='title%04d' % i, source='synthetic_game' if i % 2 else 'synthetic_movie',\n                         score=score, text=text, bucket=int(rng.integers(0, 10)),\n                         label=0 if score < 6 else 1 if score <= 7.5 else 2))\n    return docs, dict(note='synthetic local test', docs=n)\n\n\n# ---------------------------------------------------------------- tokenizer\n\nclass Tokenizer:\n    def __init__(self):\n        self.vocab = {'<pad>': 0, '<unk>': 1}\n\n    def fit(self, texts, min_df=2, cap=8000):\n        cnt = Counter()\n        for t in texts:\n            cnt.update(set(text_key(t).split()))\n        words = sorted(((w, c) for w, c in cnt.items() if c >= min_df), key=lambda wc: (-wc[1], wc[0]))\n        for w, _ in words[:cap]:\n            self.vocab[w] = len(self.vocab)\n        return self\n\n    def encode(self, text, L=MAX_TOKENS):\n        ids = [self.vocab.get(w, 1) for w in text_key(text).split()][:L]\n        return np.array(ids + [0] * (L - len(ids)), dtype=np.int64)\n\n    def encode_many(self, texts, L=MAX_TOKENS):\n        return np.stack([self.encode(t, L) for t in texts])\n\n\n# ---------------------------------------------------------------- TextCNN\n\nclass TextCNN:\n    \"\"\"Kim-style 1D CNN over token embeddings with max-over-time pooling and a\n    linear score-bucket head. Pure NumPy forward/backward.\"\"\"\n\n    def __init__(self, vocab, de=24, sizes=(3, 4, 5), nf=16, classes=3, seed=SEED, dt=np.float32):\n        rng = np.random.default_rng(seed)\n        self.de, self.sizes, self.nf, self.classes = de, sizes, nf, classes\n        self.E = rng.normal(0, 0.1, (vocab, de)).astype(dt)\n        self.W = [(rng.normal(0, 1, (nf, k, de)) / np.sqrt(k * de)).astype(dt) for k in sizes]\n        self.cb = [np.zeros(nf, dt) for _ in sizes]\n        self.Wh = (rng.normal(0, 1, (nf * len(sizes), classes)) / np.sqrt(nf * len(sizes))).astype(dt)\n        self.bh = np.zeros(classes, dt)\n\n    def params(self):\n        return [self.E] + list(self.W) + list(self.cb) + [self.Wh, self.bh]\n\n    def forward(self, x):\n        B, L = x.shape\n        emb = self.E[x]\n        zs = []; cache = []\n        for W, b in zip(self.W, self.cb):\n            k = W.shape[1]; P = L - k + 1\n            cols = np.stack([emb[:, i:i + P, :] for i in range(k)], axis=2).reshape(B * P, k * self.de)\n            out = (cols @ W.reshape(self.nf, k * self.de).T + b).reshape(B, P, self.nf)\n            act = np.maximum(out, 0)\n            zs.append(act.max(axis=1))\n            cache.append((cols, W, act, act.argmax(axis=1)))\n        z = np.concatenate(zs, axis=1)\n        logits = z @ self.Wh + self.bh\n        return logits, z, cache, emb\n\n    def loss_grads(self, x, y):\n        B, L = x.shape\n        logits, z, cache, emb = self.forward(x)\n        p = np.exp(logits - logits.max(-1, keepdims=True)); p /= p.sum(-1, keepdims=True)\n        loss = float(-np.log(p[np.arange(B), y] + 1e-12).mean())\n        dlogits = p; dlogits[np.arange(B), y] -= 1; dlogits /= B\n        gWh = z.T @ dlogits; gbh = dlogits.sum(0)\n        dz = dlogits @ self.Wh.T\n        gE = np.zeros_like(self.E); gW = []; gcb = []\n        rows = np.arange(B)[:, None]; cols_nf = np.arange(self.nf)[None, :]\n        for s, (cols, W, act, am) in enumerate(cache):\n            k = W.shape[1]; P = act.shape[1]\n            dm = dz[:, s * self.nf:(s + 1) * self.nf]\n            dpool = np.zeros((B, P, self.nf), dz.dtype)\n            dpool[rows, am, cols_nf] = dm\n            dout = dpool * (act > 0)\n            dout2 = dout.reshape(B * P, self.nf)\n            gW.append((dout2.T @ cols).reshape(self.nf, k, self.de))\n            gcb.append(dout2.sum(0))\n            dcols = (dout2 @ W.reshape(self.nf, k * self.de)).reshape(B, P, k, self.de)\n            demb = np.zeros_like(emb)\n            for i in range(k):\n                demb[:, i:i + P, :] += dcols[:, :, i, :]\n            np.add.at(gE, x.ravel(), demb.reshape(B * L, self.de))\n        return loss, [gE] + gW + gcb + [gWh, gbh]\n\n    def fit(self, X, y, Xv, yv, epochs=20, batch=64, lr=0.02, seed=SEED):\n        rng = np.random.default_rng(seed)\n        opt = Adam(self.params(), lr=lr)\n        history = []\n        n = len(X)\n        for ep in range(epochs):\n            order = rng.permutation(n)\n            total = 0.0\n            for i in range(0, n, batch):\n                idx = order[i:i + batch]\n                loss, grads = self.loss_grads(X[idx], y[idx])\n                opt.step(grads)\n                total += loss * len(idx)\n            acc = self.accuracy(Xv, yv)\n            history.append(dict(epoch=ep, train_loss=round(total / n, 6), val_accuracy=round(acc, 6)))\n            print('CNN epoch', ep, 'loss %.4f' % (total / n), 'val_acc %.4f' % acc, flush=True)\n        return history\n\n    def predict(self, X, batch=512):\n        out = []\n        for i in range(0, len(X), batch):\n            logits, _, _, _ = self.forward(X[i:i + batch])\n            out.append(logits.argmax(-1))\n        return np.concatenate(out)\n\n    def accuracy(self, X, y):\n        return float((self.predict(X) == y).mean())\n\n    def embed(self, X, batch=512):\n        out = []\n        for i in range(0, len(X), batch):\n            _, z, _, _ = self.forward(X[i:i + batch])\n            out.append(z)\n        z = np.concatenate(out).astype(np.float32)\n        return z / np.maximum(np.linalg.norm(z, axis=1, keepdims=True), 1e-12)\n\n\ndef macro_f1(y, p, classes=3):\n    f1s = []\n    for c in range(classes):\n        tp = float(((p == c) & (y == c)).sum()); fp = float(((p == c) & (y != c)).sum()); fn = float(((p != c) & (y == c)).sum())\n        f1s.append(2 * tp / max(2 * tp + fp + fn, 1))\n    return float(np.mean(f1s))\n\n\n# ---------------------------------------------------------------- mini GPT\n\ndef ln_fwd(x, g, b, eps=1e-5):\n    mu = x.mean(-1, keepdims=True); var = ((x - mu) ** 2).mean(-1, keepdims=True)\n    s = 1.0 / np.sqrt(var + eps)\n    return (x - mu) * s * g + b, (x - mu, s, g)\n\n\ndef ln_bwd(dy, cache):\n    u, s, g = cache\n    N = u.shape[-1]\n    gdy = g * dy\n    dg = (dy * u * s).sum((0, 1)); db = dy.sum((0, 1))\n    dx = (s / N) * (N * gdy - gdy.sum(-1, keepdims=True) - u * (s * s) * (gdy * u).sum(-1, keepdims=True))\n    return dx, dg, db\n\n\ndef gelu(x):\n    return 0.5 * x * (1 + np.tanh(0.7978845608 * (x + 0.044715 * x ** 3)))\n\n\ndef gelu_grad(x):\n    inner = 0.7978845608 * (x + 0.044715 * x ** 3)\n    t = np.tanh(inner)\n    return 0.5 * (1 + t) + 0.5 * x * (1 - t ** 2) * 0.7978845608 * (1 + 3 * 0.044715 * x ** 2)\n\n\ndef softmax(z, axis=-1):\n    z = z - z.max(axis=axis, keepdims=True)\n    e = np.exp(z)\n    return e / e.sum(axis=axis, keepdims=True)\n\n\nclass MiniGPT:\n    \"\"\"Small decoder-only transformer (GPT-style causal LM) in pure NumPy.\"\"\"\n\n    def __init__(self, vocab, L=MAX_TOKENS, d=48, heads=2, layers=2, ff=96, seed=SEED, dt=np.float32):\n        rng = np.random.default_rng(seed)\n        self.L, self.d, self.H, self.layers_n, self.ff = L, d, heads, layers, ff\n        self.dh = d // heads\n        self.tok = rng.normal(0, 0.05, (vocab, d)).astype(dt)\n        self.pos = rng.normal(0, 0.02, (L, d)).astype(dt)\n        self.layers = []\n        for _ in range(layers):\n            self.layers.append(dict(\n                g1=np.ones(d, dt), b1=np.zeros(d, dt),\n                Wq=(rng.normal(0, 1, (d, d)) / np.sqrt(d)).astype(dt),\n                Wk=(rng.normal(0, 1, (d, d)) / np.sqrt(d)).astype(dt),\n                Wv=(rng.normal(0, 1, (d, d)) / np.sqrt(d)).astype(dt),\n                Wo=(rng.normal(0, 1, (d, d)) / np.sqrt(d)).astype(dt),\n                g2=np.ones(d, dt), b2=np.zeros(d, dt),\n                W1=(rng.normal(0, 1, (d, ff)) / np.sqrt(d)).astype(dt),\n                bf1=np.zeros(ff, dt),\n                W2=(rng.normal(0, 1, (ff, d)) / np.sqrt(ff)).astype(dt),\n                bf2=np.zeros(d, dt)))\n        self.gf = np.ones(d, dt); self.bf = np.zeros(d, dt)\n        self.head = rng.normal(0, 0.05, (d, vocab)).astype(dt)\n\n    def params(self):\n        ps = [self.tok, self.pos]\n        for l in self.layers:\n            ps += [l[k] for k in ('g1', 'b1', 'Wq', 'Wk', 'Wv', 'Wo', 'g2', 'b2', 'W1', 'bf1', 'W2', 'bf2')]\n        ps += [self.gf, self.bf, self.head]\n        return ps\n\n    def forward(self, x):\n        B, L = x.shape\n        H, dh = self.H, self.dh\n        mask = np.triu(np.full((L, L), -1e9, np.float32), k=1)\n        h = self.tok[x] + self.pos[None, :, :]\n        caches = []\n        for l in self.layers:\n            a, c1 = ln_fwd(h, l['g1'], l['b1'])\n            q = (a @ l['Wq']).reshape(B, L, H, dh).transpose(0, 2, 1, 3)\n            k = (a @ l['Wk']).reshape(B, L, H, dh).transpose(0, 2, 1, 3)\n            v = (a @ l['Wv']).reshape(B, L, H, dh).transpose(0, 2, 1, 3)\n            scores = q @ k.transpose(0, 1, 3, 2) / np.sqrt(dh) + mask[None, None]\n            attn = softmax(scores, -1)\n            ctx = (attn @ v).transpose(0, 2, 1, 3).reshape(B, L, self.d)\n            out = ctx @ l['Wo']\n            h1 = h + out\n            a2, c2 = ln_fwd(h1, l['g2'], l['b2'])\n            pre = a2 @ l['W1'] + l['bf1']\n            f = gelu(pre)\n            h2 = h1 + f @ l['W2'] + l['bf2']\n            caches.append((c1, a, q, k, v, attn, ctx, c2, a2, pre, f))\n            h = h2\n        hf, cf = ln_fwd(h, self.gf, self.bf)\n        logits = hf @ self.head\n        return logits, (h, hf, caches, cf, x)\n\n    def loss_grads(self, x):\n        B, L = x.shape\n        V = self.head.shape[1]\n        logits, (h, hf, caches, cf, _) = self.forward(x)\n        targets = np.concatenate([x[:, 1:], np.zeros((B, 1), np.int64)], axis=1)\n        p = softmax(logits, -1)\n        flat_p = p.reshape(B * L, V)\n        loss = float(-np.log(flat_p[np.arange(B * L), targets.reshape(-1)] + 1e-12).mean())\n        dlogits = p.copy()\n        dlogits.reshape(B * L, V)[np.arange(B * L), targets.reshape(-1)] -= 1\n        dlogits /= B * L\n        ghead = hf.reshape(-1, self.d).T @ dlogits.reshape(-1, V)\n        dh = dlogits.reshape(B, L, V) @ self.head.T\n        dh, dgf, dbf = ln_bwd(dh, cf)\n        layer_grads = []\n        for li in range(self.layers_n - 1, -1, -1):\n            l = self.layers[li]\n            c1, a, q, k, v, attn, ctx, c2, a2, pre, f = caches[li]\n            # FFN branch (h2 = h1 + f @ W2 + bf2)\n            df2 = dh\n            gW2 = f.reshape(-1, self.ff).T @ df2.reshape(-1, self.d)\n            gbf2 = df2.sum((0, 1))\n            dpre = (df2 @ l['W2'].T) * gelu_grad(pre)\n            gW1 = a2.reshape(-1, self.d).T @ dpre.reshape(-1, self.ff)\n            gbf1 = dpre.sum((0, 1))\n            da2 = dpre @ l['W1'].T\n            dx2, dg2, db2 = ln_bwd(da2, c2)\n            dh1 = dh + dx2\n            # attention branch (h1 = h + ctx @ Wo)\n            dout = dh1\n            gWo = ctx.reshape(-1, self.d).T @ dout.reshape(-1, self.d)\n            dctx = (dout @ l['Wo'].T).reshape(B, L, self.H, self.dh).transpose(0, 2, 1, 3)\n            dv = attn.transpose(0, 1, 3, 2) @ dctx\n            dattn = dctx @ v.transpose(0, 1, 3, 2)\n            dscores = attn * (dattn - (dattn * attn).sum(-1, keepdims=True)) / np.sqrt(self.dh)\n            dq = (dscores @ k).transpose(0, 2, 1, 3).reshape(B, L, self.d)\n            dk = (dscores.transpose(0, 1, 3, 2) @ q).transpose(0, 2, 1, 3).reshape(B, L, self.d)\n            dv = dv.transpose(0, 2, 1, 3).reshape(B, L, self.d)\n            da = dq @ l['Wq'].T + dk @ l['Wk'].T + dv @ l['Wv'].T\n            gWq = a.reshape(-1, self.d).T @ dq.reshape(-1, self.d)\n            gWk = a.reshape(-1, self.d).T @ dk.reshape(-1, self.d)\n            gWv = a.reshape(-1, self.d).T @ dv.reshape(-1, self.d)\n            da_ln = ln_bwd(da, c1)\n            dh = dh1 + da_ln[0]\n            layer_grads.append((da_ln[1], da_ln[2], gWq, gWk, gWv, gWo, dg2, db2, gW1, gbf1, gW2, gbf2))\n        layer_grads.reverse()\n        gtok = np.zeros_like(self.tok)\n        np.add.at(gtok, x.ravel(), dh.reshape(B * L, self.d))\n        gpos = dh.sum(0)\n        grads = [gtok, gpos]\n        for g in layer_grads:\n            grads += list(g)\n        grads += [dgf, dbf, ghead]\n        return loss, grads\n\n    def nll(self, X, batch=64):\n        L = X.shape[1]\n        V = self.head.shape[1]\n        total = 0.0; count = 0\n        for i in range(0, len(X), batch):\n            x = X[i:i + batch]\n            b = len(x)\n            logits, _ = self.forward(x)\n            targets = np.concatenate([x[:, 1:], np.zeros((b, 1), np.int64)], axis=1)\n            p = softmax(logits, -1)\n            total += float(-np.log(p.reshape(b * L, V)[np.arange(b * L), targets.reshape(-1)] + 1e-12).sum())\n            count += b * L\n        return total / max(count, 1)\n\n    def fit(self, X, Xv, steps=1500, batch=16, lr=0.005, seed=SEED, log_every=250):\n        rng = np.random.default_rng(seed)\n        opt = Adam(self.params(), lr=lr)\n        history = []\n        n = len(X)\n        for s in range(steps):\n            idx = rng.integers(0, n, size=batch)\n            loss, grads = self.loss_grads(X[idx])\n            opt.step(grads)\n            if (s + 1) % log_every == 0:\n                ppl = math.exp(self.nll(Xv[:256]))\n                history.append(dict(step=s + 1, train_loss=round(loss, 6), val_perplexity=round(ppl, 4)))\n                print('GPT step', s + 1, 'loss %.4f' % loss, 'val_ppl %.2f' % ppl, flush=True)\n        return history\n\n    def embed(self, X, batch=128):\n        out = []\n        for i in range(0, len(X), batch):\n            _, (h, _, _, _, _) = self.forward(X[i:i + batch])\n            out.append(h.mean(axis=1))\n        z = np.concatenate(out).astype(np.float32)\n        return z / np.maximum(np.linalg.norm(z, axis=1, keepdims=True), 1e-12)\n\n\ndef unigram_perplexity(train_texts, val_texts, L=MAX_TOKENS):\n    cnt = Counter(); total = 0\n    for t in train_texts:\n        ws = text_key(t).split()[:L]\n        cnt.update(ws); total += len(ws)\n    V = len(cnt) + 1\n    logp = {w: math.log((c + 1) / (total + V)) for w, c in cnt.items()}\n    default = math.log(1.0 / (total + V))\n    nll = 0.0; n = 0\n    for t in val_texts:\n        for w in text_key(t).split()[:L]:\n            nll -= logp.get(w, default); n += 1\n    return math.exp(nll / max(n, 1))\n\n\ndef svd_encoder(train_texts, all_texts, dim=64):\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    from sklearn.decomposition import TruncatedSVD\n    tf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=20000, min_df=2, sublinear_tf=True)\n    Xtr = tf.fit_transform(train_texts)\n    svd = TruncatedSVD(n_components=dim, n_iter=5, random_state=SEED)\n    svd.fit(Xtr)\n    Z = svd.transform(tf.transform(all_texts))\n    Z = Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-12)\n    return Z.astype(np.float32), float(svd.explained_variance_ratio_.sum()), len(tf.vocabulary_)\n\n\n# ---------------------------------------------------------------- preference classifier\n\ndef build_stats(train, dim=16, epochs=30):\n    from distran_ml.baseline import fit\n    users = sorted({e['user_id'] for e in train})\n    items = sorted({e['item_id'] for e in train})\n    ui = {u: i for i, u in enumerate(users)}; ii = {x: j for j, x in enumerate(items)}\n    i_pos = Counter(); i_cnt = Counter(); i_sum = defaultdict(float); i_last = {}\n    u_cnt = Counter(); u_pos = Counter(); u_last = {}; u_dom = defaultdict(Counter)\n    for e in train:\n        r = e['rating']; u = e['user_id']; it = e['item_id']; t = e['timestamp']; d = it.split(':')[0]\n        i_cnt[it] += 1; i_sum[it] += r; i_pos[it] += int(r >= 4); i_last[it] = max(i_last.get(it, 0), t)\n        u_cnt[u] += 1; u_pos[u] += int(r >= 4); u_last[u] = max(u_last.get(u, 0), t); u_dom[u][d] += int(r >= 4)\n    max_t = max(e['timestamp'] for e in train)\n    pop_sorted = np.sort(np.array([i_pos[i] for i in items], dtype=float))\n    i_vec = np.zeros((len(items), 5), np.float32)\n    for j, it in enumerate(items):\n        dom = it.split(':')[0]\n        age_days = max(0.0, (max_t - i_last.get(it, max_t)) / 86400000.0)\n        i_vec[j] = [np.searchsorted(pop_sorted, i_pos[it], side='right') / len(pop_sorted),\n                    (i_sum[it] / i_cnt[it]) / 5.0,\n                    1.0 if dom == 'game' else 0.0, 1.0 if dom == 'screen' else 0.0,\n                    2.0 ** (-age_days / 90.0)]\n    max_cnt = max(u_cnt.values())\n    u_vec = np.zeros((len(users), 5), np.float32)\n    aff = np.zeros((len(users), 2), np.float32)\n    for i, u in enumerate(users):\n        tot_pos = max(1, u_pos[u])\n        aff[i] = [u_dom[u]['game'] / tot_pos, u_dom[u]['screen'] / tot_pos]\n        u_vec[i] = [math.log1p(u_cnt[u]) / math.log1p(max_cnt), u_pos[u] / u_cnt[u],\n                    aff[i, 0], aff[i, 1],\n                    2.0 ** (-max(0.0, (max_t - u_last.get(u, max_t)) / 86400000.0) / 30.0)]\n    print('Fitting BPR item factors on train for the collaborative feature', flush=True)\n    b_users, b_items, U, V = fit(train, dim=dim, epochs=epochs, seed=SEED)\n    assert b_users == users and b_items == items\n    collab_pct = np.zeros((len(users), len(items)), np.float32)\n    for i in range(len(users)):\n        row = U[i] @ V.T\n        collab_pct[i] = np.searchsorted(np.sort(row), row, side='right') / len(row)\n    # Cold items (present only in validation/test) get a neutral item vector:\n    # zero popularity, global mean quality, domain one-hots, no recency, median collab.\n    cold_quality = float(np.mean([i_sum[i] / i_cnt[i] for i in items]) / 5.0)\n    return dict(users=users, items=items, ui=ui, ii=ii, i_vec=i_vec, u_vec=u_vec, aff=aff,\n                collab_pct=collab_pct, u_cnt=u_cnt, cold_quality=cold_quality)\n\n\ndef user_item_matrix(stats, i):\n    \"\"\"Feature matrix over all items for user index i (n_items x len(FEATURES)).\"\"\"\n    X = np.empty((len(stats['items']), len(FEATURES)), np.float32)\n    X[:, :5] = stats['i_vec']\n    X[:, 5:10] = stats['u_vec'][i]\n    X[:, 10] = stats['aff'][i, 0] * stats['i_vec'][:, 2]\n    X[:, 11] = stats['aff'][i, 1] * stats['i_vec'][:, 3]\n    X[:, 12] = stats['collab_pct'][i]\n    return X\n\n\ndef cold_i_vec(it, quality):\n    dom = it.split(':')[0]\n    return np.array([0.0, quality, 1.0 if dom == 'game' else 0.0,\n                     1.0 if dom == 'screen' else 0.0, 0.0], np.float32)\n\n\ndef event_features(stats, u, it):\n    i = stats['ui'][u]; j = stats['ii'].get(it)\n    iv = stats['i_vec'][j] if j is not None else cold_i_vec(it, stats['cold_quality'])\n    collab = stats['collab_pct'][i, j] if j is not None else 0.5\n    return np.concatenate([iv, stats['u_vec'][i],\n                           [stats['aff'][i, 0] * iv[2], stats['aff'][i, 1] * iv[3], collab]]).astype(np.float32)\n\n\ndef score_rows(stats, theta, b, state=None):\n    rows = {}\n    for i, u in enumerate(stats['users']):\n        delta, bu = (state or {}).get(u, (np.zeros(len(FEATURES), np.float32), 0.0))\n        rows[u] = sigmoid(b + bu + user_item_matrix(stats, i) @ (theta + delta)).astype(np.float32)\n    return rows\n\n\ndef train_shared_theta(train, stats, epochs=6, neg_per_user=5, lr=0.01, seed=SEED):\n    rng = np.random.default_rng(seed)\n    seen = defaultdict(set)\n    for e in train:\n        seen[e['user_id']].add(e['item_id'])\n    Xs = []; ys = []\n    for e in train:\n        Xs.append(event_features(stats, e['user_id'], e['item_id']))\n        ys.append(1.0 if e['rating'] >= 4 else 0.0)\n    items = stats['items']\n    for u in stats['users']:\n        negs = [it for it in items if it not in seen[u]]\n        if not negs:\n            continue\n        for it in rng.choice(negs, size=min(neg_per_user, len(negs)), replace=False):\n            Xs.append(event_features(stats, u, it)); ys.append(0.0)\n    X = np.stack(Xs); y = np.array(ys, np.float32)\n    theta = np.zeros(len(FEATURES), np.float32); b = np.array(0.0, dtype=np.float32)\n    opt = Adam([theta, b], lr=lr)\n    n = len(X)\n    for ep in range(epochs):\n        order = rng.permutation(n)\n        total = 0.0\n        for i in range(0, n, 512):\n            idx = order[i:i + 512]\n            p = sigmoid(b + X[idx] @ theta)\n            g = (X[idx].T @ (p - y[idx]) / len(idx)).astype(np.float32)\n            gb = np.array(float((p - y[idx]).mean()), dtype=np.float32)\n            opt.step([g, gb])\n            total += float(-(y[idx] * np.log(p + 1e-12) + (1 - y[idx]) * np.log(1 - p + 1e-12)).mean()) * len(idx)\n        print('shared theta epoch', ep, 'logloss %.4f' % (total / n), flush=True)\n    return theta, float(b)\n\n\ndef replay(events, stats, theta, b, lr_u, reg_u, state=None):\n    \"\"\"Chronological predict-before-learn replay of the per-user residual.\n\n    p(u,i,c) = sigmoid(b + b_u + (theta + delta_u) . x); after each event\n    delta_u += lr_u * clip_norm((y - p) * x) - lr_u * reg_u * delta_u.\n    \"\"\"\n    state = dict(state or {})\n    preds = []\n    for e in sorted(events, key=lambda e: e['timestamp']):\n        u = e['user_id']\n        if u not in stats['ui']:  # validation/test-only user: neutral profile\n            x = np.concatenate([cold_i_vec(e['item_id'], stats['cold_quality']),\n                                np.zeros(5, np.float32),\n                                [0.0, 0.0, 0.5]]).astype(np.float32)\n        else:\n            x = event_features(stats, u, e['item_id'])\n        delta, bu = state.get(u, (np.zeros(len(FEATURES), np.float32), 0.0))\n        p = float(sigmoid(b + bu + (theta + delta) @ x))\n        preds.append(dict(user_id=u, item_id=e['item_id'], rating=e['rating'], p=p))\n        y = 1.0 if e['rating'] >= 4 else 0.0\n        upd = lr_u * (y - p) * x\n        norm = float(np.linalg.norm(upd))\n        if norm > 1.0:\n            upd = upd / norm\n        delta = np.clip(delta + upd - lr_u * reg_u * delta, -10, 10)\n        bu = float(np.clip(bu + lr_u * (y - p), -10, 10))\n        state[u] = (delta, bu)\n    return preds, state\n\n\ndef pointwise(preds):\n    y = np.array([1.0 if p['rating'] >= 4 else 0.0 for p in preds])\n    q = np.array([p['p'] for p in preds])\n    eps = 1e-12\n    logloss = float(-(y * np.log(q + eps) + (1 - y) * np.log(1 - q + eps)).mean())\n    brier = float(((y - q) ** 2).mean())\n    bins = []\n    edges = np.linspace(0, 1, 11)\n    for lo, hi in zip(edges[:-1], edges[1:]):\n        m = (q >= lo) & (q < hi) if hi < 1 else (q >= lo) & (q <= hi)\n        if m.sum():\n            bins.append(dict(range=[float(lo), float(hi)], n=int(m.sum()),\n                             mean_pred=round(float(q[m].mean()), 4), observed=round(float(y[m].mean()), 4)))\n    return dict(logloss=round(logloss, 6), brier=round(brier, 6), n=len(preds), calibration=bins)\n\n\ndef rank_eval(score_fn, stats, train, heldout, domain=None, source=None, k=10, seed=SEED):\n    users, items = stats['users'], stats['items']\n    ii = stats['ii']\n    seen = defaultdict(set); relevant = defaultdict(set); positive_domains = defaultdict(set)\n    for e in train:\n        seen[e['user_id']].add(e['item_id'])\n        if e['rating'] >= 4:\n            positive_domains[e['user_id']].add(e['item_id'].split(':')[0])\n    for e in heldout:\n        if e['rating'] >= 4 and (domain is None or e['item_id'].startswith(domain + ':')):\n            relevant[e['user_id']].add(e['item_id'])\n    domain_mask = np.ones(len(items), bool) if domain is None else np.array([x.startswith(domain + ':') for x in items])\n    rows = []; recommended = set()\n    item_set = set(items)\n    cold_users = 0; cold_item_positives = 0\n    for u, truth in relevant.items():\n        cold_item_positives += len(truth - item_set)\n        if source and source not in positive_domains[u]:\n            continue\n        if u not in stats['ui']:\n            cold_users += 1\n            continue\n        truth = (truth & item_set) - seen[u]\n        if not truth:\n            continue\n        scores = score_fn(u)\n        mask = domain_mask.copy()\n        if seen[u]:\n            mask[[ii[x] for x in seen[u]]] = False\n        candidates = np.nonzero(mask)[0]\n        ranked = candidates[np.argsort(-scores[candidates], kind='stable')[:k]]\n        recommended.update(items[j] for j in ranked)\n        hits = np.array([items[j] in truth for j in ranked], dtype=float)\n        recall = float(hits.sum() / len(truth))\n        ideal = sum(1 / np.log2(r + 2) for r in range(min(k, len(truth))))\n        ndcg = float(sum(h / np.log2(r + 2) for r, h in enumerate(hits)) / ideal)\n        rows.append((recall, ndcg))\n    a = np.array(rows)\n    metrics = dict(evaluated_users=len(rows), positive_heldout_users=len(relevant), cold_heldout_users=cold_users,\n                   cold_item_positives=cold_item_positives, k=k,\n                   ranking='full train catalog, exclude all training interactions',\n                   catalog_coverage=len(recommended) / max(1, int(domain_mask.sum())))\n    for j, key in enumerate(['recall_at_10', 'ndcg_at_10']):\n        metrics[key] = float(a[:, j].mean()) if len(rows) else None\n        if len(rows):\n            rng = np.random.default_rng(seed)\n            boots = [float(a[rng.integers(0, len(a), len(a)), j].mean()) for _ in range(400)]\n            metrics[key + '_ci95'] = [round(float(x), 6) for x in np.quantile(boots, [.025, .975])]\n    return metrics\n\n\nCOHORTS = [('all', None, None), ('game', 'game', None), ('screen', 'screen', None),\n           ('game_to_screen', 'screen', 'game'), ('screen_to_game', 'game', 'screen')]\n\n\ndef evaluate_models(models, stats, train, heldout):\n    results = {}\n    for name, fn in models.items():\n        results[name] = {cohort: rank_eval(fn, stats, train, heldout, domain, source)\n                         for cohort, domain, source in COHORTS}\n        print('Evaluated', name, flush=True)\n    return results\n\n\n# ---------------------------------------------------------------- staged run\n\ndef _load_ctx(root):\n    import pickle\n    cache = root / 'cache'; cache.mkdir(exist_ok=True, parents=True)\n    ckpt = cache / 'checkpoint.json'\n    done = set(json.loads(ckpt.read_text())['done']) if ckpt.exists() else set()\n    ctx = {}\n    pkl = cache / 'ctx.pkl'\n    if pkl.exists():\n        ctx = pickle.loads(pkl.read_bytes())\n    return cache, done, ctx\n\n\ndef _save_ctx(cache, done, ctx):\n    import pickle\n    (cache / 'ctx.pkl').write_bytes(pickle.dumps(ctx))\n    (cache / 'checkpoint.json').write_text(json.dumps({'version': MODULE_VERSION, 'done': sorted(done)}))\n\n\ndef run(root=Path('/content/distran-adaptive')):\n    \"\"\"Staged, checkpointed run: each stage persists to root/cache so an\n    interrupted Colab cell can resume on the next `colab exec` call.\"\"\"\n    started = time.time()\n    root = Path(root); output = root / 'artifacts'; output.mkdir(exist_ok=True, parents=True)\n    local_test = os.environ.get('ADAPTIVE_LOCAL_TEST') == '1'\n    cache, done, ctx = _load_ctx(root)\n    # invalidate checkpoints written by older module versions\n    done.discard('encoders')\n    if cache.joinpath('checkpoint.json').exists():\n        try:\n            old_version = json.loads(cache.joinpath('checkpoint.json').read_text()).get('version')\n        except Exception:\n            old_version = None\n        if old_version != MODULE_VERSION:\n            for k in ('grid', 'test', 'export'):\n                done.discard(k)\n            if old_version is None or old_version < 2:\n                for k in ('stats', 'theta'):\n                    done.discard(k)\n    rng = np.random.default_rng(SEED)\n\n    def stage(name):\n        if name in done:\n            print('stage', name, 'cached', flush=True)\n            return False\n        print('stage', name, 'running', flush=True)\n        return True\n\n    if local_test:\n        catalog = [dict(id='synthetic-%02d' % i, title='title%04d' % (i * 37 % 1200),\n                        medium='game' if i % 2 else 'movie',\n                        description='synthetic work description number %d with some words' % i,\n                        tags=['a', 'b'], maker='studio') for i in range(12)]\n        (root / 'catalog-metadata.json').write_text(json.dumps(catalog))\n    else:\n        catalog = json.loads((root / 'catalog-metadata.json').read_text())\n    v2 = json.loads((root / 'catalog-model-v2.json').read_text()) if (root / 'catalog-model-v2.json').exists() else None\n\n    # stage 1/2: Amazon download and parse\n    if local_test:\n        if stage('parse_amazon'):\n            ctx['events'], ctx['event_audit'] = synthetic_events(rng)\n            from distran_ml.baseline import temporal_split\n            ctx['train'], ctx['val'], ctx['test'] = temporal_split(ctx['events'])\n            print('Split', len(ctx['train']), len(ctx['val']), len(ctx['test']), flush=True)\n            done.add('parse_amazon'); _save_ctx(cache, done, ctx)\n    else:\n      if stage('dl_amazon'):\n        salt = os.urandom(32)\n        sources = []\n        for category in ('Video_Games', 'Movies_and_TV'):\n            path = root / (category + '.csv.gz'); url = AMAZON_BASE + path.name\n            print('Downloading complete file:', category, flush=True)\n            if not path.exists():\n                with urllib.request.urlopen(url, timeout=180) as response, path.open('wb') as out:\n                    while chunk := response.read(1024 * 1024):\n                        out.write(chunk)\n            sources.append(dict(url=url, sha256=hashlib.sha256(path.read_bytes()).hexdigest(),\n                                bytes=path.stat().st_size))\n        ctx['amazon_sources'] = sources; ctx['salt'] = salt\n        done.add('dl_amazon'); _save_ctx(cache, done, ctx)\n      if stage('parse_amazon'):\n        events = []; user_sets = []\n        for domain, category in [('game', 'Video_Games'), ('screen', 'Movies_and_TV')]:\n            path = root / (category + '.csv.gz')\n            users = set(); count = 0; kept = 0\n            with gzip.open(path, 'rt') as stream:\n                for row in csv.DictReader(stream):\n                    count += 1; raw = row['user_id']; users.add(raw)\n                    if int.from_bytes(hashlib.sha256(raw.encode()).digest()[:8], 'big') % 128:\n                        continue\n                    rating = float(row['rating']); timestamp = int(row['timestamp'])\n                    if not np.isfinite(rating) or not 1 <= rating <= 5:\n                        raise ValueError('Invalid rating')\n                    user = hashlib.sha256(ctx['salt'] + raw.encode()).hexdigest()\n                    events.append(dict(user_id=user, item_id=domain + ':' + row['parent_asin'],\n                                       rating=rating, timestamp=timestamp)); kept += 1\n            user_sets.append(users)\n            ctx['amazon_sources'][[s['url'] for s in ctx['amazon_sources']].index(AMAZON_BASE + path.name)].update(\n                rows=count, users=len(users), sampled_rows=kept)\n            print(category, 'rows', count, 'sampled', kept, flush=True)\n        overlap = len(user_sets[0] & user_sets[1]); del user_sets\n        unique = {(e['user_id'], e['item_id'], e['timestamp']): e for e in events}\n        ctx['events'] = list(unique.values())\n        ctx['event_audit'] = dict(sources=ctx['amazon_sources'], full_file_overlap_users=overlap,\n                                  duplicates_removed=len(events) - len(unique),\n                                  sample='SHA256(source_user_id) first 8 bytes big endian modulo 128 == 0; complete file scans; no future-based user filtering',\n                                  pseudonyms='Random per-run salt, not exported; sampled cohort deterministic, pseudonyms differ each run',\n                                  limitations='Publisher 5-core selection uses complete history; screen combines movies and TV. Same sampling rule as notebook 03 so cohorts are comparable.')\n        from distran_ml.baseline import temporal_split\n        ctx['train'], ctx['val'], ctx['test'] = temporal_split(ctx['events'])\n        print('Split', len(ctx['train']), len(ctx['val']), len(ctx['test']), flush=True)\n        done.add('parse_amazon'); _save_ctx(cache, done, ctx)\n\n    # stage 3/4: Kaggle corpus download and build\n    if local_test:\n        if stage('build_corpus'):\n            ctx['docs'], ctx['corpus_audit'] = synthetic_corpus(rng)\n            done.add('build_corpus'); _save_ctx(cache, done, ctx)\n    else:\n      if stage('dl_corpus'):\n        import kagglehub\n        ctx['kaggle_sources'] = {}\n        for key, handle in KAGGLE.items():\n            directory = Path(kagglehub.dataset_download(handle)); path = directory / KAGGLE_FILES[key]\n            ctx['kaggle_sources'][key] = dict(handle=handle, file=path.name,\n                                              sha256=hashlib.sha256(path.read_bytes()).hexdigest(),\n                                              bytes=path.stat().st_size, resolved=str(directory))\n            print('Kaggle', key, ctx['kaggle_sources'][key]['sha256'][:16], ctx['kaggle_sources'][key]['bytes'], flush=True)\n        done.add('dl_corpus'); _save_ctx(cache, done, ctx)\n      if stage('build_corpus'):\n        import kagglehub\n        import pandas as pd\n        docs = []\n        directory = Path(kagglehub.dataset_download(KAGGLE['movies']))\n        raw = pd.read_csv(directory / KAGGLE_FILES['movies'], low_memory=False)\n        ctx['kaggle_sources']['movies']['rows'] = len(raw)\n        for _, row in raw.iterrows():\n            title = clean(row.get('title'))\n            if not title:\n                continue\n            votes = float(pd.to_numeric(pd.Series([row.get('vote_count')]), errors='coerce').iloc[0])\n            score = float(pd.to_numeric(pd.Series([row.get('vote_average')]), errors='coerce').iloc[0])\n            score = score if np.isfinite(score) and np.isfinite(votes) and votes >= 5 and 0 <= score <= 10 else None\n            docs.append(dict(title=title, source='tmdb', score=score,\n                             text=title + '. ' + clean(row.get('overview')) + ' language_' + clean(row.get('original_lang'))))\n        directory = Path(kagglehub.dataset_download(KAGGLE['hltb']))\n        hltb = pd.read_csv(directory / KAGGLE_FILES['hltb'], low_memory=False)\n        ctx['kaggle_sources']['hltb']['rows'] = len(hltb)\n        hltb = hltb[hltb['game_game_name'].notna()].copy()\n        hltb = hltb.sort_values(['game_game_id', 'platform_platform'], kind='stable').drop_duplicates('game_game_id')\n        hltb['order'] = hltb['game_game_name'].map(lambda t: hashlib.sha256(title_key(t).encode()).hexdigest())\n        hltb = hltb.sort_values('order', kind='stable').head(HLTB_SAMPLE)\n        for _, row in hltb.iterrows():\n            title = clean(row.get('game_game_name'))\n            reviews = float(pd.to_numeric(pd.Series([row.get('game_count_review')]), errors='coerce').iloc[0])\n            score = float(pd.to_numeric(pd.Series([row.get('game_review_score')]), errors='coerce').iloc[0])\n            score = score / 10.0 if np.isfinite(score) and np.isfinite(reviews) and reviews >= 5 and 0 < score <= 100 else None\n            doc = ' '.join(clean(row.get(c)) for c in ('game_profile_summary', 'game_profile_genre', 'game_profile_dev', 'game_profile_pub'))\n            docs.append(dict(title=title, source='hltb', score=score, text=title + '. ' + doc))\n        for d in docs:\n            d['bucket'] = split_bucket(d['title'])\n            d['label'] = None if d['score'] is None else (0 if d['score'] < 6 else 1 if d['score'] <= 7.5 else 2)\n        ctx['docs'] = docs\n        ctx['corpus_audit'] = dict(sources=ctx['kaggle_sources'], docs=len(docs),\n                                   scored=sum(d['label'] is not None for d in docs), hltb_sample=HLTB_SAMPLE,\n                                   hltb_policy='deterministic SHA256-title-ordered subset; one row per game_game_id',\n                                   split='SHA256(normalized title)[:8] hex mod 10; buckets 0-5 train, 6-7 validation, 8-9 test',\n                                   score_policy='TMDB vote_average (0-10) with vote_count>=5; HLTB game_review_score/10 with game_count_review>=5; buckets low<6, mid 6-7.5, high>7.5')\n        done.add('build_corpus'); _save_ctx(cache, done, ctx)\n\n    docs = ctx['docs']; train, val, test = ctx['train'], ctx['val'], ctx['test']\n\n    # stage 5a: tokenizer + TextCNN encoder\n    if stage('cnn'):\n        tok = Tokenizer().fit([d['text'] for d in docs if d['bucket'] < 6])\n        X_all = tok.encode_many([d['text'] for d in docs])\n        tr = [i for i, d in enumerate(docs) if d['bucket'] < 6]\n        va = [i for i, d in enumerate(docs) if 6 <= d['bucket'] < 8]\n        te = [i for i, d in enumerate(docs) if d['bucket'] >= 8]\n        scored = lambda idxs: [i for i in idxs if docs[i]['label'] is not None]\n        str_, sva, ste = scored(tr), scored(va), scored(te)\n        print('corpus', len(docs), 'vocab', len(tok.vocab), 'scored', len(str_) + len(sva) + len(ste),\n              'docs train/val/test', len(tr), len(va), len(te), flush=True)\n        cnn = TextCNN(len(tok.vocab))\n        ytr = np.array([docs[i]['label'] for i in str_]); yva = np.array([docs[i]['label'] for i in sva])\n        yte = np.array([docs[i]['label'] for i in ste])\n        cnn_hist = cnn.fit(X_all[str_], ytr, X_all[sva], yva, epochs=3 if local_test else 15)\n        cnn_pred = cnn.predict(X_all[ste])\n        cnn_report = dict(architecture='TextCNN: 24-d token embeddings, conv1d filter sizes 3/4/5 with 16 filters each, ReLU, max-over-time pooling, linear 3-class score-bucket head; NumPy Adam',\n                          params=int(sum(p.size for p in cnn.params())), train_epochs=len(cnn_hist),\n                          test_accuracy=round(cnn.accuracy(X_all[ste], yte), 6), test_macro_f1=round(macro_f1(yte, cnn_pred), 6),\n                          majority_baseline=round(Counter(yte.tolist()).most_common(1)[0][1] / len(yte), 6),\n                          history=cnn_hist[-3:])\n        print('CNN test acc', cnn_report['test_accuracy'], 'f1', cnn_report['test_macro_f1'],\n              'baseline', cnn_report['majority_baseline'], flush=True)\n        ctx.update(tok=tok, X_all=X_all, tr=tr, va=va, te=te, str_=str_, sva=sva, ste=ste,\n                   ytr=ytr, yva=yva, yte=yte, cnn=cnn, cnn_report=cnn_report)\n        done.add('cnn'); _save_ctx(cache, done, ctx)\n\n    # stage 5b: mini-GPT encoder\n    if stage('gpt'):\n        tok, X_all = ctx['tok'], ctx['X_all']\n        tr, va = ctx['tr'], ctx['va']\n        gpt = MiniGPT(len(tok.vocab))\n        gpt_hist = gpt.fit(X_all[tr], X_all[va], steps=120 if local_test else 1200,\n                           log_every=40 if local_test else 200)\n        gpt_ppl = math.exp(gpt.nll(X_all[va]))\n        uni_ppl = unigram_perplexity([docs[i]['text'] for i in tr], [docs[i]['text'] for i in va])\n        gpt_report = dict(architecture='decoder-only transformer (GPT-style): d=48, 2 heads, 2 pre-LN blocks, ff=96, causal next-token LM over 48-token windows; NumPy Adam',\n                          params=int(sum(p.size for p in gpt.params())), steps=gpt_hist[-1]['step'],\n                          val_perplexity=round(gpt_ppl, 4), unigram_baseline_perplexity=round(uni_ppl, 4),\n                          history=gpt_hist[-3:])\n        print('GPT val ppl', gpt_report['val_perplexity'], 'unigram', gpt_report['unigram_baseline_perplexity'], flush=True)\n        ctx.update(gpt=gpt, gpt_report=gpt_report)\n        done.add('gpt'); _save_ctx(cache, done, ctx)\n\n    # stage 5c: SVD baseline, corpus embeddings, curated-work probes\n    if stage('compare'):\n        tok, X_all = ctx['tok'], ctx['X_all']\n        tr = ctx['tr']\n        cnn, gpt = ctx['cnn'], ctx['gpt']\n        Z_svd, svd_var, svd_vocab = svd_encoder([docs[i]['text'] for i in tr], [d['text'] for d in docs])\n        Z_cnn = cnn.embed(X_all); Z_gpt = gpt.embed(X_all)\n        sample = rng.choice(len(docs), size=min(2000, len(docs)), replace=False)\n        agreement = {'cnn_gpt_mean_cosine': round(float((Z_cnn[sample] * Z_gpt[sample]).sum(1).mean()), 6)}\n        curated_text = [' '.join([w['title'], clean(w.get('description')), ' '.join(w.get('tags', [])), clean(w.get('maker'))]) for w in catalog]\n        Xc = tok.encode_many(curated_text)\n        Zc_svd, _, _ = svd_encoder([docs[i]['text'] for i in tr], curated_text)\n        P = {'cnn': cnn.embed(Xc), 'gpt': gpt.embed(Xc), 'svd': Zc_svd}\n        if v2 is not None:\n            Z = np.array(v2['vectors'], np.float32)\n            P['svd_v2_app'] = Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-12)\n        probes = {}\n        for enc, Z in P.items():\n            probes[enc] = [dict(seed=w['id'], neighbors=[catalog[j]['id'] for j in np.argsort(-(Z @ Z[n])) if j != n][:3])\n                           for n, w in enumerate(catalog)]\n        for i, a in enumerate(list(P)):\n            for b in list(P)[i + 1:]:\n                ja = [len(set(pa['neighbors']) & set(pb['neighbors'])) / max(1, len(set(pa['neighbors']) | set(pb['neighbors'])))\n                      for pa, pb in zip(probes[a], probes[b])]\n                agreement[f'probe_top3_jaccard_{a}_{b}'] = round(float(np.mean(ja)), 6)\n        ctx.update(Z_svd=Z_svd, svd_var=svd_var, svd_vocab=svd_vocab, agreement=agreement, probes=probes)\n        done.add('compare'); _save_ctx(cache, done, ctx)\n\n    # stage 6: interaction feature stats + BPR collaborative feature\n    if stage('stats'):\n        ctx['stats'] = build_stats(train)\n        done.add('stats'); _save_ctx(cache, done, ctx)\n\n    # stage 7: shared logistic prior\n    if stage('theta'):\n        ctx['theta'], ctx['b'] = train_shared_theta(train, ctx['stats'])\n        done.add('theta'); _save_ctx(cache, done, ctx)\n\n    # stage 8: per-user residual hyperparameter selection on validation (logloss only;\n    # one validation ranking for the selected point, to keep the stage bounded)\n    if stage('grid'):\n        stats = ctx['stats']; theta, b = ctx['theta'], ctx['b']\n        grid = [(lr, reg) for lr in (0.05, 0.1) for reg in (0.0, 0.01)]\n        validation = {}; best = None\n        for lr_u, reg_u in grid:\n            _, state = replay(train, stats, theta, b, lr_u, reg_u)\n            vp, _ = replay(val, stats, theta, b, lr_u, reg_u, state=dict(state))\n            pw = pointwise(vp)\n            validation[f'lr={lr_u},reg={reg_u}'] = dict(logloss=pw['logloss'])\n            print('adaptive grid', lr_u, reg_u, 'val logloss %.4f' % pw['logloss'], flush=True)\n            if best is None or pw['logloss'] < validation[best]['logloss']:\n                best = f'lr={lr_u},reg={reg_u}'\n        ctx['validation'] = validation; ctx['best'] = best\n        ctx['lr_u'], ctx['reg_u'] = (float(x.split('=')[1]) for x in best.split(','))\n        _, ctx['state'] = replay(train, stats, theta, b, ctx['lr_u'], ctx['reg_u'])\n        rows = score_rows(stats, theta, b, ctx['state'])\n        ctx['validation'][best]['ndcg_at_10'] = rank_eval(lambda u: rows[u], stats, train, val)['ndcg_at_10']\n        done.add('grid'); _save_ctx(cache, done, ctx)\n\n    # stage 9: heldout test evaluation, once, after selection\n    if stage('test'):\n        stats = ctx['stats']; theta, b = ctx['theta'], ctx['b']; state = ctx['state']\n        tp, _ = replay(test, stats, theta, b, ctx['lr_u'], ctx['reg_u'], state=dict(state))\n        sp, _ = replay(test, stats, theta, b, 0.0, 0.0)\n        tpw = pointwise(tp); spw = pointwise(sp)\n        print('selected', ctx['best'], 'test logloss adaptive %.4f shared %.4f' % (tpw['logloss'], spw['logloss']), flush=True)\n        fixed_theta = np.array([1.0, 0.8, 0.0, 0.0, 0.1, 0.2, 0.5, 0.0, 0.0, 0.1, 1.5, 1.5, 1.2], np.float32)\n        fixed_rows = {u: sigmoid(-2.0 + user_item_matrix(stats, i) @ fixed_theta).astype(np.float32)\n                      for i, u in enumerate(stats['users'])}\n        shared_rows = score_rows(stats, theta, b)\n        adaptive_rows = score_rows(stats, theta, b, state)\n        pop_row = stats['i_vec'][:, 0]\n        models = {\n            'popularity': lambda u: pop_row,\n            'bpr': lambda u: stats['collab_pct'][stats['ui'][u]],\n            'fixed_linear_heuristic': lambda u: fixed_rows[u],\n            'shared_logistic': lambda u: shared_rows[u],\n            'adaptive': lambda u: adaptive_rows[u],\n        }\n        ctx['test_results'] = evaluate_models(models, stats, train, test)\n        ctx['val_results'] = evaluate_models({'adaptive': lambda u: adaptive_rows[u]}, stats, train, val)\n        ctx['tpw'] = tpw; ctx['spw'] = spw\n        done.add('test'); _save_ctx(cache, done, ctx)\n\n    # stage 10: safe exports, report, provenance, zip\n    if stage('export'):\n        stats = ctx['stats']; theta, b = ctx['theta'], ctx['b']; state = ctx['state']\n        salt = os.urandom(32)\n        deltas_export = {}\n        for u, (delta, bu) in state.items():\n            if stats['u_cnt'][u] >= 10:\n                deltas_export[hashlib.sha256(salt + u.encode()).hexdigest()[:16]] = dict(\n                    delta=[round(float(x), 6) for x in delta], b_u=round(bu, 6))\n            if len(deltas_export) >= 5000:\n                break\n        coefficients = dict(schema=1, model_id=MODEL_ID, name=MODEL_NAME,\n                            feature_schema=FEATURES, theta=[round(float(x), 6) for x in theta], intercept=round(b, 6),\n                            selected=dict(lr_u=ctx['lr_u'], reg_u=ctx['reg_u'], criterion='validation logloss under predict-before-learn replay'),\n                            update_rule='delta_u += lr_u * clip_norm((y - p) * x, 1) - lr_u * reg_u * delta_u; b_u += lr_u * (y - p); both clipped to [-10, 10]',\n                            label_policy='rating >= 4 positive, rating <= 3 negative; unobserved items are sampled negatives for the shared prior only, never proven dislikes',\n                            per_user_residuals=deltas_export,\n                            status='research export; salted user keys; not promoted to the app')\n        (output / 'adaptive-coefficients.json').write_text(json.dumps(coefficients, indent=2))\n        np.savez_compressed(output / 'encoders.npz', cnn_E=ctx['cnn'].E,\n                            cnn_W=np.concatenate([w.reshape(-1) for w in ctx['cnn'].W]), cnn_Wh=ctx['cnn'].Wh, cnn_bh=ctx['cnn'].bh,\n                            gpt_tok=ctx['gpt'].tok, gpt_pos=ctx['gpt'].pos, gpt_head=ctx['gpt'].head,\n                            Z_cnn=ctx['cnn'].embed(ctx['X_all']), Z_gpt=ctx['gpt'].embed(ctx['X_all']), Z_svd=ctx['Z_svd'])\n        (output / 'tokenizer.json').write_text(json.dumps(ctx['tok'].vocab))\n        (output / 'catalog-probes.json').write_text(json.dumps(ctx['probes'], indent=2))\n        report = dict(run_id='adaptive-personalization-20261122',\n                      runtime='local synthetic test' if local_test else 'Google Colab CPU via colab CLI (staged, checkpointed cells)',\n                      seed=SEED,\n                      model_card=dict(name=MODEL_NAME, model_id=MODEL_ID,\n                                      classification=dict(\n                                          task='binary preference classification plus ranking (NDCG/Recall@10) with log loss, Brier and calibration',\n                                          family='shared logistic prior theta plus private per-user residual delta_u, online SGD with chronological predict-before-learn replay; feature/score fusion, not parameter averaging',\n                                          encoders={'textcnn': ctx['cnn_report']['architecture'], 'mini_gpt': ctx['gpt_report']['architecture'],\n                                                    'baseline': 'TF-IDF + truncated SVD (notebook 04 shared encoder)'},\n                                          serving='local JSON coefficients per account; no hosted GPU service',\n                                          status='research only; not promoted')),\n                      environment=dict(python=sys.version.split()[0], numpy=np.__version__, platform=platform.platform()),\n                      events=ctx['event_audit'],\n                      counts=dict(events=len(ctx['events']), train=len(train), validation=len(val), test=len(test),\n                                  users=len(stats['users']), items=len(stats['items'])),\n                      corpus=ctx['corpus_audit'],\n                      encoders=dict(textcnn=ctx['cnn_report'], mini_gpt=ctx['gpt_report'],\n                                    svd=dict(dim=64, explained_variance=round(ctx['svd_var'], 6), vocabulary=ctx['svd_vocab']),\n                                    agreement=ctx['agreement'], probes=ctx['probes']),\n                      classifier=dict(features=FEATURES, theta=[round(float(x), 6) for x in theta], intercept=round(b, 6),\n                                      validation=ctx['validation'], selected=ctx['best'],\n                                      test_pointwise=dict(adaptive=ctx['tpw'], shared_only=ctx['spw']),\n                                      validation_ranking=ctx['val_results'], test=ctx['test_results']),\n                      parameters=dict(bpr=dict(dim=16, epochs=30, positive='rating >= 4'),\n                                      shared_theta=dict(epochs=6, neg_per_user=5),\n                                      adaptive=dict(lr_u=ctx['lr_u'], reg_u=ctx['reg_u'], half_life_user_days=30,\n                                                    half_life_item_days=90, clip=10)),\n                      seconds=round(time.time() - started, 2),\n                      limitations=['Amazon IDs have no reviewed mapping to the app catalog; the CNN/GPT semantic channels are demonstrated on the metadata corpus and curated probes, not as features inside the Amazon replay.',\n                                   'Features are as-of the train split (static within each split); the online replay is predict-before-learn on labels but not expanding-window on features.',\n                                   'Corpus splits are title-group hash splits (snapshot), while the interaction split is chronological; score buckets are aggregate quality, not personal taste.',\n                                   'Single fixed test measurement per model; bootstrap CIs are user-level only and no cross-model significance is claimed.',\n                                   'No impression ledger or logged propensities exist, so no bandit or off-policy evaluation is possible.'])\n        (output / 'report.json').write_text(json.dumps(report, indent=2))\n        (output / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))\n        nb_hash = (root / 'notebook.sha256').read_text().strip() if (root / 'notebook.sha256').exists() else None\n        provenance = dict(run_id=report['run_id'], notebook_sha256=nb_hash,\n                          amazon_sources=ctx['event_audit'].get('sources'),\n                          kaggle_sources=ctx['corpus_audit'].get('sources'),\n                          catalog_sha256=hashlib.sha256((root / 'catalog-metadata.json').read_bytes()).hexdigest(),\n                          experiment_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())\n        (output / 'provenance.json').write_text(json.dumps(provenance, indent=2))\n        print(json.dumps(ctx['tpw'], indent=2)[:1500], flush=True)\n        import zipfile\n        with zipfile.ZipFile(root / 'results.zip', 'w', zipfile.ZIP_DEFLATED) as z:\n            for file in output.iterdir():\n                z.write(file, file.name)\n        done.add('export'); _save_ctx(cache, done, ctx)\n        ctx['report'] = report\n    return ctx.get('report')\n\n\nif __name__ == '__main__':\n    run(Path(sys.argv[1]) if len(sys.argv) > 1 else Path('/content/distran-adaptive'))\n",
    "distran_ml/baseline.py": "\"\"\"Small, inspectable BPR experiment. CPU/Colab; not production serving.\"\"\"\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\nimport numpy as np\n\n\ndef temporal_split(events):\n    \"\"\"Global time cutoffs prevent one user's future leaking into another's past.\"\"\"\n    times = sorted({int(e['timestamp']) for e in events})\n    if len(times) < 3:\n        raise ValueError('Need at least three distinct timestamps')\n    ai = min(len(times)-3, max(0, int(len(times)*.6)-1))\n    bi = min(len(times)-2, max(ai+1, int(len(times)*.8)-1))\n    a, b = times[ai], times[bi]\n    return ([e for e in events if e['timestamp'] <= a],\n            [e for e in events if a < e['timestamp'] <= b],\n            [e for e in events if e['timestamp'] > b])\n\n\ndef fit(events, dim=16, epochs=30, seed=42, lr=.03, reg=.01):\n    users = sorted({e['user_id'] for e in events})\n    items = sorted({e['item_id'] for e in events})\n    ui, ii = {x:i for i,x in enumerate(users)}, {x:i for i,x in enumerate(items)}\n    positives = defaultdict(set)\n    for e in events:\n        if e['rating'] >= 4:\n            positives[ui[e['user_id']]].add(ii[e['item_id']])\n    rng = np.random.default_rng(seed)\n    U = rng.normal(0,.1,(len(users),dim)); V = rng.normal(0,.1,(len(items),dim))\n    # Complement precomputed for small experiments only. Replace at large scale.\n    # Keep complements as arrays: rng.choice(list) converts the entire list\n    # on every SGD step, making even bounded real-data runs unnecessarily slow.\n    negatives = {u: np.array(sorted(set(range(len(items)))-seen), dtype=np.int64)\n                 for u,seen in positives.items()}\n    pairs = [(u,i) for u,seen in positives.items() for i in sorted(seen) if len(negatives[u])]\n    if not pairs:\n        raise ValueError('No trainable positive/negative pairs')\n    for _ in range(epochs):\n        for idx in rng.permutation(len(pairs)):\n            u,i = pairs[idx]; j = int(rng.choice(negatives[u]))\n            old = U[u].copy(); vi=V[i].copy(); vj=V[j].copy()\n            g = 1/(1+np.exp(np.clip(old @ (vi-vj), -30, 30)))\n            U[u] += lr*(g*(vi-vj)-reg*old)\n            V[i] += lr*(g*old-reg*vi); V[j] += lr*(-g*old-reg*vj)\n    return users, items, U, V\n\n\ndef evaluate(model, train, heldout, k=10):\n    users,items,U,V=model; ui={u:i for i,u in enumerate(users)}\n    seen=defaultdict(set); relevant=defaultdict(set)\n    for e in train: seen[e['user_id']].add(e['item_id'])\n    for e in heldout:\n        if e['rating']>=4: relevant[e['user_id']].add(e['item_id'])\n    recalls=[]; ndcgs=[]; eligible=0\n    for u,truth in relevant.items():\n        if u not in ui: continue\n        truth=truth.intersection(items)-seen[u]\n        if not truth: continue\n        eligible+=1\n        ranked=sorted((i for i in range(len(items)) if items[i] not in seen[u]), key=lambda i: (-float(U[ui[u]]@V[i]),items[i]))[:k]\n        hits=[int(items[i] in truth) for i in ranked]\n        recalls.append(sum(hits)/len(truth))\n        dcg=sum(h/np.log2(r+2) for r,h in enumerate(hits))\n        ideal=sum(1/np.log2(r+2) for r in range(min(k,len(truth))))\n        ndcgs.append(dcg/ideal)\n    return {'recall_at_k': float(np.mean(recalls)) if recalls else None,\n            'ndcg_at_k': float(np.mean(ndcgs)) if ndcgs else None,\n            'evaluated_users':eligible,'positive_heldout_users':len(relevant), 'k':k}\n\n\ndef demo():\n    events=[]\n    for t in range(1,31):\n        for u in range(12):\n            item=(u*3+t//2)%36\n            events.append(dict(user_id=f'demo:{u}',item_id=f'{\"game\" if item%2 else \"movie\"}:{item}',rating=5 if item%3==u%3 else 3,timestamp=t))\n    train,val,test=temporal_split(events)\n    model=fit(train)\n    return model, {'validation':evaluate(model,train,val),'test':evaluate(model,train,test),'synthetic':True}\n\nif __name__=='__main__':\n    model,metrics=demo(); print(json.dumps(metrics,indent=2))\n",
    "catalog-metadata.json": "[\n  {\n    \"id\": \"it-takes-two\",\n    \"title\": \"It Takes Two\",\n    \"year\": 2021,\n    \"medium\": \"game\",\n    \"description\": \"Two people, one extraordinary world. Play as Cody and May, a couple transformed into dolls, on a wonderfully unpredictable journey built entirely for two.\",\n    \"tags\": [\n      \"Co-op\",\n      \"Adventure\",\n      \"A little wonder\"\n    ],\n    \"maker\": \"Hazelight Studios\"\n  },\n  {\n    \"id\": \"cyberpunk\",\n    \"title\": \"Cyberpunk 2077\",\n    \"year\": 2020,\n    \"medium\": \"game\",\n    \"description\": \"Become V, a mercenary searching for a one-of-a-kind implant in Night City. A story about identity, impossible choices, and the people you find at the edge of the future.\",\n    \"tags\": [\n      \"Cyberpunk\",\n      \"Open world\",\n      \"Night city\"\n    ],\n    \"maker\": \"CD PROJEKT RED\"\n  },\n  {\n    \"id\": \"elden-ring\",\n    \"title\": \"Elden Ring\",\n    \"year\": 2022,\n    \"medium\": \"game\",\n    \"description\": \"Rise, Tarnished, and follow the grace of gold into the Lands Between. A vast, mysterious world where every ruin has a story and every distant light is an invitation.\",\n    \"tags\": [\n      \"Dark fantasy\",\n      \"Exploration\",\n      \"RPG\"\n    ],\n    \"maker\": \"FromSoftware\"\n  },\n  {\n    \"id\": \"disco-elysium\",\n    \"title\": \"Disco Elysium\",\n    \"year\": 2019,\n    \"medium\": \"game\",\n    \"description\": \"You are a detective with a city to unravel and a mind that will not stay quiet. Question the world, question yourself, and find something worth holding on to in Revachol.\",\n    \"tags\": [\n      \"Detective\",\n      \"Narrative\",\n      \"Weird cities\"\n    ],\n    \"maker\": \"ZA/UM\"\n  },\n  {\n    \"id\": \"baldurs-gate\",\n    \"title\": \"Baldur\u2019s Gate 3\",\n    \"year\": 2023,\n    \"medium\": \"game\",\n    \"description\": \"Gather your party and return to the Forgotten Realms. An adventure shaped by friendship, freedom, and the choices that make a story yours.\",\n    \"tags\": [\n      \"RPG\",\n      \"Co-op\",\n      \"Fantasy\"\n    ],\n    \"maker\": \"Larian Studios\"\n  },\n  {\n    \"id\": \"outer-wilds\",\n    \"title\": \"Outer Wilds\",\n    \"year\": 2019,\n    \"medium\": \"game\",\n    \"description\": \"An open-world mystery about a solar system trapped in an endless time loop. Pack your curiosity, follow an unfamiliar signal, and discover what waits beyond the next horizon.\",\n    \"tags\": [\n      \"Exploration\",\n      \"Space\",\n      \"Mystery\"\n    ],\n    \"maker\": \"Mobius Digital\"\n  },\n  {\n    \"id\": \"exit-8\",\n    \"title\": \"The Exit 8\",\n    \"year\": 2023,\n    \"medium\": \"game\",\n    \"description\": \"You are trapped in an endless underground passageway. Observe your surroundings carefully. If you find an anomaly, turn back. Find the way to Exit 8.\",\n    \"tags\": [\n      \"Liminal\",\n      \"Horror\",\n      \"Short journey\"\n    ],\n    \"maker\": \"KOTAKE CREATE\"\n  },\n  {\n    \"id\": \"dune\",\n    \"title\": \"Dune: Part Two\",\n    \"year\": 2024,\n    \"medium\": \"movie\",\n    \"description\": \"Paul Atreides joins Chani and the Fremen while seeking revenge against the conspirators who destroyed his family. In the desert, a personal journey becomes something much larger.\",\n    \"tags\": [\n      \"Science fiction\",\n      \"Epic\",\n      \"World-building\"\n    ],\n    \"maker\": \"Denis Villeneuve\"\n  },\n  {\n    \"id\": \"blade-runner\",\n    \"title\": \"Blade Runner 2049\",\n    \"year\": 2017,\n    \"medium\": \"movie\",\n    \"description\": \"A new blade runner uncovers a secret that could change what remains of society. His search leads through a rain-soaked future, toward questions about memory and being human.\",\n    \"tags\": [\n      \"Cyberpunk\",\n      \"Atmospheric\",\n      \"Identity\"\n    ],\n    \"maker\": \"Denis Villeneuve\"\n  },\n  {\n    \"id\": \"arrival\",\n    \"title\": \"Arrival\",\n    \"year\": 2016,\n    \"medium\": \"movie\",\n    \"description\": \"When mysterious spacecraft appear across the world, a linguist searches for a way to communicate. A quiet, moving story about language, time, and the connections that shape a life.\",\n    \"tags\": [\n      \"Science fiction\",\n      \"Language\",\n      \"Contemplative\"\n    ],\n    \"maker\": \"Denis Villeneuve\"\n  },\n  {\n    \"id\": \"severance\",\n    \"title\": \"Severance\",\n    \"year\": 2022,\n    \"medium\": \"tv\",\n    \"description\": \"A team of office workers have surgically divided their memories between work and personal life. When the boundary begins to unravel, they discover that nothing about their work is ordinary.\",\n    \"tags\": [\n      \"Mystery\",\n      \"Identity\",\n      \"Uncanny\"\n    ],\n    \"maker\": \"Dan Erickson\"\n  },\n  {\n    \"id\": \"edgerunners\",\n    \"title\": \"Cyberpunk: Edgerunners\",\n    \"year\": 2022,\n    \"medium\": \"tv\",\n    \"description\": \"A street kid tries to survive in a city obsessed with technology and body modification. A standalone story in the world of Cyberpunk 2077, about found family and the price of a dream.\",\n    \"tags\": [\n      \"Anime\",\n      \"Cyberpunk\",\n      \"Found family\"\n    ],\n    \"maker\": \"Studio Trigger\"\n  }\n]",
    "catalog-model-v2.json": "{\n  \"schema\": 1,\n  \"model_id\": \"kaggle-catalog-tfidf-svd-v2\",\n  \"feature_schema\": \"title-description-tags-maker-v2\",\n  \"catalog_sha256\": \"d01bf1bd975f4ae2951e61a6ec9b9b3c1ba9cc4a94f70e3de8d8148806411988\",\n  \"items\": [\n    \"it-takes-two\",\n    \"cyberpunk\",\n    \"elden-ring\",\n    \"disco-elysium\",\n    \"baldurs-gate\",\n    \"outer-wilds\",\n    \"exit-8\",\n    \"dune\",\n    \"blade-runner\",\n    \"arrival\",\n    \"severance\",\n    \"edgerunners\"\n  ],\n  \"media\": [\n    \"game\",\n    \"game\",\n    \"game\",\n    \"game\",\n    \"game\",\n    \"game\",\n    \"game\",\n    \"movie\",\n    \"movie\",\n    \"movie\",\n    \"tv\",\n    \"tv\"\n  ],\n  \"vectors\": [\n    [\n      0.17541093006430192,\n      0.05385388328584754,\n      0.058642787410352254,\n      0.1058424097128131,\n      0.004300779959183388,\n      0.4820578095103842,\n      -0.0897846825381283,\n      -0.10368963001438242,\n      0.04204448135938963,\n      0.33829237157034275,\n      0.19970409204963835,\n      0.08238334822395453,\n      0.13839882643412865,\n      -0.04884615480560511,\n      -0.08623806893700839,\n      0.040825522958777886,\n      -0.14805843690874135,\n      -0.05507842550075924,\n      0.039585332430660085,\n      -0.09786718952349846,\n      -0.005922975760503382,\n      -0.0046431594792737734,\n      -0.20243576332184413,\n      -0.04799773771486709,\n      -0.10126569319199236,\n      -0.039440058652293344,\n      0.01283299745410401,\n      0.038074405421961374,\n      0.039673122392920646,\n      -0.15428177841018792,\n      0.004938910081795824,\n      0.16236475361491354,\n      0.3848626198536051,\n      0.18622311941613068,\n      -0.04135571747132134,\n      -0.19762224749936289,\n      -0.005764757483574443,\n      0.0070681224321637575,\n      0.14957917436811782,\n      -0.04619895337419781,\n      -0.005870434077148932,\n      -0.008879125477664871,\n      -0.1384361468836343,\n      -0.14749910825179258,\n      -0.01581809697307415,\n      0.09121195867845365,\n      -0.0838364299130101,\n      0.01672171608985248,\n      0.044970370678820135,\n      0.014887634174020167,\n      0.04276465707035675,\n      -0.04756015295293607,\n      0.04090684499733532,\n      -0.025087674140070786,\n      -0.04099098821017758,\n      -0.04276186333800142,\n      0.0141027329015642,\n      0.1856665256582385,\n      -0.029549784956520903,\n      -0.004043202078453788,\n      -0.07730463689814081,\n      0.04668931491346111,\n      0.028418528941148012,\n      -0.02429978613356725\n    ],\n    [\n      0.12216676404816128,\n      0.02029238991218536,\n      0.055224174584160284,\n      0.10785591980549054,\n      0.007158957925811264,\n      0.3453257148986945,\n      -0.2295187948713064,\n      0.11251181526364899,\n      -0.009274655230206107,\n      -0.015347930720468006,\n      -0.0003140841297503312,\n      -0.013858858952799744,\n      0.002331690459249838,\n      -0.00683236774084307,\n      0.03215266800516688,\n      -0.0159497985034286,\n      0.013333134932819738,\n      -0.02097623327086277,\n      0.024109821473156295,\n      0.015234557544240662,\n      0.005880368488701591,\n      -0.05004882367672193,\n      -0.02517359797888962,\n      0.006158825577713131,\n      -0.020647335829340786,\n      0.10518031709379995,\n      -0.0073063253326425505,\n      -0.014928405192413062,\n      0.017090477675622596,\n      0.13308213460040527,\n      -0.047681351371251644,\n      0.09165318766633339,\n      0.30627707571014684,\n      0.15799189581136877,\n      -0.05682561998510102,\n      -0.14448031614870507,\n      -0.03475885827447287,\n      0.008483552218474872,\n      0.16938631872409926,\n      0.026667750217909573,\n      -0.01871101599656342,\n      -0.00036519959214552817,\n      -0.13902984891542636,\n      -0.22619763347816793,\n      -0.040697319852780314,\n      -0.13192893868038844,\n      0.041285739848496934,\n      0.3399805381824413,\n      -0.011208428690818584,\n      0.09339020382264879,\n      0.06500386115298469,\n      0.08795357103498487,\n      -0.06278468861726791,\n      -0.21630252787866286,\n      0.028892540908936712,\n      -0.11072856384875401,\n      0.02816159048877652,\n      -0.1784206071704429,\n      -0.11977428366806783,\n      -0.09620452152585796,\n      0.09059539101120262,\n      0.17842290523868806,\n      0.4163808934551373,\n      0.05809887398847396\n    ],\n    [\n      0.14432809355722076,\n      0.02542656923240267,\n      0.045423699878140676,\n      0.19550844144888035,\n      0.027965114169986004,\n      0.31589994800333454,\n      -0.19619012025979926,\n      0.08557643750632256,\n      -0.018681516620104362,\n      0.00635259341609915,\n      0.031168128545014002,\n      -0.0012464935914571458,\n      -0.008890673915491781,\n      -0.043067563218096926,\n      -0.00733822401514783,\n      -0.02726582701788318,\n      0.0680443435347291,\n      -0.0324165182548962,\n      -0.10040434285751614,\n      -0.09104377625544548,\n      0.030541536323682703,\n      -0.04092440825111497,\n      -0.04961879062713531,\n      0.010137475814748205,\n      -0.0395673847691405,\n      0.1669132033918225,\n      0.002168836005002484,\n      -0.05999351346945344,\n      0.006163933838371462,\n      0.09808949283294455,\n      -0.06178954675343215,\n      0.11348745057298358,\n      0.43890811980891686,\n      0.17679328791819005,\n      -0.07899003206062252,\n      -0.14298174133087718,\n      0.013160619956595754,\n      0.010790433444282951,\n      0.30568451690545295,\n      0.02502717604047086,\n      0.01594824811253064,\n      -0.021541700513805308,\n      -0.23105637497184658,\n      -0.05915245470172495,\n      -0.03939303972929171,\n      0.08203796286429456,\n      -0.09994092477419562,\n      0.02209542702847475,\n      0.033304312408893345,\n      0.008525957468843284,\n      0.03442889197651949,\n      -0.06472055052248238,\n      -0.10374689389751654,\n      -0.039626354033332076,\n      0.013409098014391713,\n      0.03155670079115384,\n      0.17924020241425156,\n      0.12558148915238704,\n      0.0868956724475829,\n      0.029572513901751432,\n      -0.26016343281516163,\n      0.005408461194838313,\n      0.34670326837156396,\n      0.11634864263216393\n    ],\n    [\n      0.10479736039399731,\n      0.01928501396848236,\n      0.047892468022577524,\n      0.07798230338949973,\n      0.009709162030876939,\n      0.3377811239695158,\n      -0.20005470539671216,\n      0.09921136180984327,\n      0.006324094133378273,\n      -0.004309688125325656,\n      0.023677170142791137,\n      0.0005713003842709282,\n      0.00481220843775111,\n      -0.001051435114437846,\n      0.036526661119260374,\n      -0.02787014119552456,\n      0.018184999038301865,\n      -0.012466838661400156,\n      -0.012669761099758769,\n      0.02069671888169173,\n      -0.011827752654304153,\n      -0.04917504920877149,\n      -0.04114447438640854,\n      0.02362153794735697,\n      -0.03351947748770823,\n      0.08833529531867956,\n      -0.010757863395581626,\n      -0.028822247634734538,\n      0.011594606370491723,\n      0.10113229670193097,\n      -0.055424726935717566,\n      0.09818162641058611,\n      0.4035491222991067,\n      0.2105523252856477,\n      -0.0667443635809035,\n      -0.2857678101575202,\n      -0.0014878397804058753,\n      -0.029993152562675485,\n      0.16004842690690327,\n      0.022263698575437724,\n      -0.04242545388732542,\n      -0.021398518621432914,\n      -0.17828564457793114,\n      -0.2605553122703945,\n      0.043032529836976856,\n      -0.10743074351032271,\n      0.019217532945051817,\n      0.40468208497578956,\n      0.023700884940195555,\n      0.08452501126056315,\n      0.06289751918942618,\n      0.11260903838287518,\n      -0.10293741150040955,\n      -0.20489786875488786,\n      -0.03941276051971375,\n      0.05360786153136568,\n      0.021149853948966714,\n      -0.10293649081255431,\n      -0.032864727678699926,\n      0.023630323207896906,\n      0.17081202241673024,\n      -0.11839797667206306,\n      0.14554426817972832,\n      0.12823004377345307\n    ],\n    [\n      0.19660435240426097,\n      0.05633526010211558,\n      0.03993220155964746,\n      0.19091000423417862,\n      0.010958291212630894,\n      0.40705340821229935,\n      -0.013736037586090103,\n      -0.12824047925846147,\n      0.052860920568896526,\n      0.39595213797489187,\n      0.17916941832673988,\n      0.07457871571564902,\n      0.15372379329047206,\n      0.006405785139500169,\n      -0.12869091448527797,\n      0.007522378036410566,\n      -0.1250057740027413,\n      -0.04590199239598356,\n      -0.04417379244986748,\n      -0.17100702051805414,\n      -0.012603557199459237,\n      -0.004439461133253011,\n      -0.20698415315645616,\n      -0.041791112437581615,\n      -0.10675121268348634,\n      -0.07850608150376387,\n      0.009007862904255617,\n      0.05202094570038807,\n      0.0605981466764889,\n      -0.24781191382991383,\n      0.026015099815562555,\n      0.09440953216725587,\n      0.1593896575432953,\n      0.051750655257255834,\n      0.017636026000555506,\n      0.046322352952311456,\n      -0.07009830021556501,\n      0.014690958347030785,\n      0.09827739712612257,\n      -0.03154185809156177,\n      0.047758736584548975,\n      0.07293024485796958,\n      -0.034283828946485205,\n      0.10752980751856665,\n      -0.13964138724440445,\n      -0.05858148343555784,\n      0.013243106408847918,\n      -0.09697059059078707,\n      0.00017516452842833885,\n      0.03799919322156267,\n      -0.09244725370598049,\n      -0.04553850328832301,\n      0.059174121053605436,\n      0.04934567374277401,\n      0.05295871374699478,\n      -0.06372806020687435,\n      0.24393986670602372,\n      0.0259629977546498,\n      0.057630951270097,\n      -0.07739916022870318,\n      -0.22856708843364154,\n      0.21789608280344763,\n      0.12380001680896202,\n      -0.05353375138698419\n    ],\n    [\n      0.21031770649464082,\n      0.18496000430533538,\n      0.02306635781042753,\n      0.01753431808366419,\n      0.23706800843388595,\n      0.3201373233753004,\n      -0.17641303594567878,\n      0.0512085356060123,\n      0.013150206104547387,\n      0.07168733692066437,\n      -0.02074565170174181,\n      -0.00752920219216162,\n      0.04322576477587844,\n      -0.08908375024219493,\n      0.02479943957709205,\n      -0.0066302215417840555,\n      0.06745290554873085,\n      -0.0045708312776413995,\n      -0.007128475883344473,\n      0.0070442969600871055,\n      0.01503001955052991,\n      -0.03271034603323264,\n      -0.07504054968317189,\n      -0.01706346761667935,\n      -0.061945066355501316,\n      0.1575941451553059,\n      -0.019386679940165793,\n      -0.09006407707623544,\n      0.0640130377365023,\n      0.1796865391903286,\n      -0.13538273233877215,\n      0.15421936596398966,\n      0.38140623968863197,\n      0.2220268185488452,\n      -0.1622242466806822,\n      -0.16925052516910385,\n      0.024332877652816106,\n      0.11633276830908314,\n      0.10444688429721882,\n      -0.049506104611948605,\n      -0.03367711515794445,\n      -0.004852791743245504,\n      -0.16368317351642214,\n      -0.15216337853094924,\n      -0.053144999411151285,\n      0.11225487880926766,\n      0.012936870919804375,\n      -0.12789246612388763,\n      -0.04832474845950445,\n      0.029628627691090557,\n      -0.08285385652022964,\n      -0.165490341382988,\n      0.06169743869914814,\n      -0.047889498737620474,\n      0.2681588824155612,\n      0.19513850265769742,\n      -0.11447482225653444,\n      0.03613866895000471,\n      -0.024538914119532188,\n      0.03181099577262901,\n      0.017818002833858083,\n      -0.21999861009286445,\n      -0.06505116565908613,\n      0.04099694367909521\n    ],\n    [\n      0.12016887413044147,\n      0.020182315416013178,\n      0.05921840124839099,\n      0.12292173032654348,\n      0.005559768956196625,\n      0.4407836833118575,\n      -0.23338516782478932,\n      0.05481037567376115,\n      0.0037605843894230615,\n      0.07311262782869163,\n      0.021318531516179257,\n      -0.0004798710965756127,\n      0.05479834004145939,\n      -0.1311242649235591,\n      0.006029182766033711,\n      -0.037614447779297726,\n      0.04956685962117393,\n      0.02745862349578152,\n      -0.0386558224729781,\n      0.009057776449703459,\n      0.0055067199618132,\n      -0.05849940043838102,\n      -0.07575327305350725,\n      -0.00786586834800043,\n      -0.043943499973357496,\n      0.21154301981214235,\n      -0.03358285982885234,\n      -0.11552269939205226,\n      0.004275515966896017,\n      0.1592875826753008,\n      -0.0951959909904461,\n      0.014247427831729083,\n      0.1767839713826928,\n      -0.0364148890261539,\n      0.03568907112439428,\n      0.06340327424366111,\n      -0.08681470319689336,\n      0.018849512704443634,\n      -0.05392877207922707,\n      -0.05004704236979145,\n      -0.026289321498324105,\n      0.07711236417247182,\n      0.04273659850010875,\n      -0.1086100101856357,\n      -0.1678048759250139,\n      0.04621578499265938,\n      0.055986598838638435,\n      -0.20884691330563035,\n      -0.054512976827281005,\n      0.004447085987985593,\n      -0.058059859126803205,\n      -0.35930245811635153,\n      0.021036536331559202,\n      -0.09110552696234199,\n      -0.005260347044806005,\n      -0.030124396210925507,\n      0.055237263088043324,\n      0.3083608594178241,\n      -0.12291109293099318,\n      -0.047345835997818844,\n      -0.13601449748182712,\n      0.2629947473103017,\n      0.2769033175801488,\n      0.06307795030269538\n    ],\n    [\n      0.12209193649339445,\n      0.028035328229532704,\n      0.058195264306213926,\n      0.09900559380289749,\n      0.007736215976366229,\n      0.4242169706267282,\n      -0.293388185513478,\n      0.16121223756203043,\n      0.0033435965653966443,\n      -0.05397952760747576,\n      0.00285978140632745,\n      0.002290307245800097,\n      0.006471130165788897,\n      0.003559250294232283,\n      0.003594320261632002,\n      0.0006766101244128044,\n      0.00867441296518583,\n      -0.03375435355024969,\n      0.002079688601390135,\n      0.00046973600770344607,\n      0.008779743854089448,\n      -0.0216658448311974,\n      -0.028166782001732116,\n      0.005174222825394527,\n      -0.03979246658717906,\n      0.08123290091333177,\n      0.023442476622389775,\n      -0.032518312979204426,\n      0.005808180652051,\n      0.052806819414672766,\n      -0.02629391947774038,\n      0.14154377350965447,\n      0.41421442141522624,\n      0.2251075572419821,\n      -0.08182941340121112,\n      -0.3528976959375163,\n      0.05544166046395729,\n      0.032308204547531354,\n      0.22801453905866806,\n      -0.003166013491225417,\n      -0.0011592273702044076,\n      -0.06955405703060936,\n      -0.1920524966469589,\n      -0.1683707356821328,\n      0.0755973246793719,\n      0.1380172382070496,\n      -0.18299916900418658,\n      0.12764220484765726,\n      -0.017877740445423174,\n      0.014872707832962849,\n      -0.013315771103831548,\n      -0.0007467420061847328,\n      -0.10682121133861269,\n      -0.1752161154871036,\n      -0.01394579530868273,\n      0.005436785544594646,\n      -0.11047317358108644,\n      0.1257753367361918,\n      0.014610878964637062,\n      -0.05925441063728068,\n      -0.0013283123931940543,\n      -0.09493922214079709,\n      -0.04405764016750804,\n      0.011378298239536953\n    ],\n    [\n      0.14085061140198585,\n      0.048242757992729236,\n      0.04936019860122766,\n      0.14375729805039653,\n      -0.003542680481250732,\n      0.5387184675541778,\n      -0.3883948347665194,\n      0.18478511364250946,\n      -0.03306489176811516,\n      -0.03653803373048364,\n      -0.02377355003905848,\n      0.003792187369647977,\n      -0.015160981914055396,\n      -0.008432472718070586,\n      -0.0035992721089199236,\n      -0.01759445095073026,\n      -0.030562109897622952,\n      -0.012919779802360856,\n      0.01213690558933591,\n      0.017767303985806112,\n      0.04081154587604547,\n      0.007143525946802317,\n      -0.019409426982388296,\n      0.006639197373942903,\n      -0.020394596905802385,\n      0.05650610279421286,\n      -0.004821228352917028,\n      0.02592552691909744,\n      -0.048597669463063736,\n      0.05084246642092008,\n      -0.04845072036155891,\n      0.007704120409687996,\n      0.03664822577119591,\n      0.07210928345239168,\n      -0.004067714372234823,\n      0.01579508238470879,\n      -0.0013995329072840847,\n      0.007172144423268216,\n      -0.055556822031029914,\n      -0.01728373331605739,\n      0.002158624647894302,\n      0.12323560666367087,\n      0.009172052182850518,\n      -0.1508482025475201,\n      -0.22248656284758778,\n      -0.21392872448679098,\n      0.2442870104714353,\n      0.4211123913977872,\n      0.09404927479860062,\n      0.008700553745132946,\n      0.06889051485852551,\n      -0.13341809904204008,\n      0.03245193549064047,\n      0.10830753671216657,\n      -0.11534818615078933,\n      0.08582632957051306,\n      -0.021089863858070913,\n      0.039789932654576615,\n      -0.07407715786236327,\n      0.001237819151906914,\n      -0.08072640553414599,\n      0.01855024801231267,\n      0.01542709177595346,\n      -0.020499429928863236\n    ],\n    [\n      0.10439808239416856,\n      0.0232622347724825,\n      0.04382041819505745,\n      0.09418624716663244,\n      0.00830761589335667,\n      0.393980870505005,\n      -0.27242086114730113,\n      0.1552913238488857,\n      0.0004280361587835292,\n      -0.033801717608911326,\n      0.004504393859033159,\n      0.0014738337821954446,\n      0.009437810736268197,\n      -0.02420441271025969,\n      0.002947833419142753,\n      -0.01428432668515885,\n      0.0020963224484324104,\n      -0.02121742110925873,\n      -0.005802809343383828,\n      0.01600469656473271,\n      0.00947838998243731,\n      -0.014502884482215954,\n      -0.03022650228996231,\n      0.0035618858108726922,\n      -0.020783160484137513,\n      0.08724401282484807,\n      -0.012139803444397863,\n      -0.03864535553721959,\n      0.015254532677683487,\n      0.12751985490708137,\n      -0.037400446233878455,\n      0.0733077503367984,\n      0.2981168946698734,\n      0.12006401173974493,\n      -0.06891861082218448,\n      -0.10442030061492226,\n      0.022961243394500907,\n      0.00034998918440606766,\n      0.15759630210315356,\n      -0.0018269669903760804,\n      -0.02239068873094564,\n      -0.04250709992932689,\n      -0.13522614449953224,\n      -0.07711801178150778,\n      0.056462243414318804,\n      0.14470614818445476,\n      -0.2500567779966839,\n      -0.06017179111372536,\n      0.39981145293952663,\n      -0.12950209395784665,\n      -0.2338763640131859,\n      -0.05888799870929622,\n      0.24743300423667572,\n      0.014422924353927542,\n      0.21854788705635686,\n      0.15648323987700388,\n      -0.10175475968117761,\n      -0.11161713322960623,\n      0.04168389062936481,\n      0.004535745741140981,\n      0.07831250492143255,\n      -0.0523821679100495,\n      0.15329434683893559,\n      -0.025170015088487752\n    ],\n    [\n      0.08765436157074225,\n      0.02118811342495344,\n      0.04380119540628176,\n      0.09863110716621598,\n      0.005205133809410545,\n      0.40952345537465334,\n      -0.3216657411115416,\n      0.18067311060976657,\n      -0.00062547409612598,\n      -0.046828092831380584,\n      -0.009782061051272967,\n      0.026371557715719682,\n      0.005090941178372797,\n      0.020987764195228034,\n      -0.0013850292113987478,\n      -0.010909602599680285,\n      -0.021312371254265286,\n      -0.006316135302099995,\n      -0.025090042619155276,\n      0.02766625728867387,\n      0.04387524989358759,\n      0.017558184701818844,\n      -0.024463793699447612,\n      -0.005660393770308577,\n      -0.01523825942065557,\n      -0.06553817069105697,\n      -0.028999720822072722,\n      0.018754324591315406,\n      0.007183061597810063,\n      0.0016679882878210773,\n      0.0108764790733623,\n      -0.05079271971150224,\n      -0.07567806866530617,\n      -0.04258384677547085,\n      0.012716267124513497,\n      0.07473153848567665,\n      -0.021138617644343754,\n      0.0072756375558468925,\n      -0.0420164878059466,\n      0.007615285389884776,\n      -0.0007631399746153494,\n      -0.02180920632069536,\n      -0.0019346697655402974,\n      -0.04894398963343486,\n      0.03231799305064791,\n      -0.007462140224541956,\n      -0.21323588736389956,\n      -0.03495309102145096,\n      0.4239832403185956,\n      -0.15275393764655734,\n      -0.1340495316636299,\n      -0.10349811508022531,\n      0.1639690557849609,\n      0.012597624265588742,\n      -0.2032304989815555,\n      0.27007743358824154,\n      0.0068012736845687125,\n      -0.21402285167785404,\n      0.25410214840254286,\n      0.2552590661056363,\n      0.07135245269750488,\n      -0.1674993318664977,\n      0.10704404763528885,\n      0.00707332604520331\n    ],\n    [\n      0.14015955049662399,\n      0.038022938294682534,\n      0.06625594320693035,\n      0.10050024193321712,\n      -0.018075395160887506,\n      0.4186835195736172,\n      -0.27740016047175287,\n      0.15553361812910657,\n      0.012674304124527599,\n      -0.04254629972974905,\n      -0.004151858720964077,\n      -0.005829208048845269,\n      0.0004424424832096093,\n      0.015546965162234982,\n      0.005546518810345825,\n      -0.005098196293867161,\n      -0.018283731715465587,\n      0.019042023584477277,\n      -0.010715833592531993,\n      -0.002291334631380722,\n      -0.01725924027897289,\n      -0.025217576739662077,\n      -0.023088010790652827,\n      0.024765266724463976,\n      -0.02345524954523667,\n      0.0740034293984133,\n      -0.012137432033958374,\n      0.009093390695592831,\n      0.01926316082029462,\n      0.11933882795946879,\n      -0.055228026512582205,\n      0.06653220239702037,\n      0.27329480212213036,\n      0.16628354667929748,\n      -0.06958513599343201,\n      -0.1408348515911679,\n      -0.003619620661475147,\n      0.007967938559443409,\n      0.14599740464606378,\n      -0.01212037594355145,\n      -0.007508611709068653,\n      -0.010532689757716845,\n      -0.13200690958950573,\n      -0.06540765806380532,\n      -0.020518888472775918,\n      -0.07623370132745028,\n      -0.003973387300026911,\n      0.2757981082016653,\n      0.09481883092835967,\n      -0.031539054470578205,\n      -0.08100603541164123,\n      0.056444594435476246,\n      -0.07395922897388,\n      -0.2780209129196413,\n      0.003941572548613912,\n      -0.14833296113098243,\n      -0.022935663794566984,\n      0.1409611531094903,\n      -0.05226557148700649,\n      -0.18521793977241133,\n      0.023350238499824975,\n      0.023787102706185764,\n      0.45150732509909325,\n      0.12012871303316214\n    ]\n  ],\n  \"fallback\": [\n    \"it-takes-two\",\n    \"cyberpunk\",\n    \"elden-ring\",\n    \"disco-elysium\",\n    \"baldurs-gate\",\n    \"outer-wilds\",\n    \"exit-8\",\n    \"dune\",\n    \"blade-runner\",\n    \"arrival\",\n    \"severance\",\n    \"edgerunners\"\n  ],\n  \"training\": {\n    \"seed\": 42,\n    \"documents\": 32952,\n    \"vocabulary\": 30000,\n    \"dimensions\": 64,\n    \"explained_variance\": 0.10591654113189175,\n    \"sources\": {\n      \"movies\": \"bc97c7426389cb652553fc6abae5943dd2174259596d40588a4c1f42d19a5f6e\",\n      \"sales\": \"4ad802a9e51792e7a96b84da4501e5f64c97484d937ede9441d2e1145c1d7b03\",\n      \"hltb\": \"63832555c3b5861b047bc095ceded72a050d4672555bb0d51850e819ced293e6\"\n    }\n  },\n  \"status\": \"preview-content-only; aggregate regressions are not personalization quality\"\n}",
}
for name, content in files.items():
    path = root / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
nb = Path("/content/05_adaptive_personalization.ipynb")
if nb.exists():
    (root / "notebook.sha256").write_text(hashlib.sha256(nb.read_bytes()).hexdigest())
print("installed", {k: len(v) for k, v in files.items()})


installed {'adaptive_experiment.py': 61179, 'distran_ml/baseline.py': 3866, 'catalog-metadata.json': 4807, 'catalog-model-v2.json': 22814}


In [ ]:
# @title Preprocess, train encoders and the adaptive classifier, select on validation, evaluate test once
import json, subprocess, sys
subprocess.check_call([sys.executable, str(__import__("pathlib").Path("/content/distran-adaptive/adaptive_experiment.py")), "/content/distran-adaptive"])
report = json.loads((__import__("pathlib").Path("/content/distran-adaptive/artifacts/report.json")).read_text())


In [ ]:
# @title Inspect measured results
c = report["classifier"]
print("selected adaptive hyperparameters:", c["selected"])
print("test pointwise adaptive:", {k: c["test_pointwise"]["adaptive"][k] for k in ("logloss", "brier", "n")})
print("test pointwise shared-only:", {k: c["test_pointwise"]["shared_only"][k] for k in ("logloss", "brier", "n")})
for model, cohorts in c["test"].items():
    print(model, {k: round(v["ndcg_at_10"], 4) if v["ndcg_at_10"] is not None else None for k, v in cohorts.items()})
e = report["encoders"]
print("textcnn:", {k: e["textcnn"][k] for k in ("test_accuracy", "test_macro_f1", "majority_baseline", "params")})
print("mini_gpt:", {k: e["mini_gpt"][k] for k in ("val_perplexity", "unigram_baseline_perplexity", "params")})
print("agreement:", e["agreement"])
print("Download /content/distran-adaptive/results.zip before stopping the runtime.")


selected adaptive hyperparameters: lr=0.1,reg=0.0
test pointwise adaptive: {'logloss': 1.745827, 'brier': 0.52121, 'n': 13195}
test pointwise shared-only: {'logloss': 2.729197, 'brier': 0.709318, 'n': 13195}
popularity {'all': 0.0, 'game': 0.0161, 'screen': 0.0, 'game_to_screen': 0.0, 'screen_to_game': 0.0}
bpr {'all': 0.0, 'game': 0.0, 'screen': 0.0, 'game_to_screen': 0.0, 'screen_to_game': 0.0}
fixed_linear_heuristic {'all': 0.0, 'game': 0.0, 'screen': 0.0, 'game_to_screen': 0.0, 'screen_to_game': 0.0}
shared_logistic {'all': 0.001, 'game': 0.012, 'screen': 0.0, 'game_to_screen': 0.0, 'screen_to_game': 0.0}
adaptive {'all': 0.001, 'game': 0.012, 'screen': 0.0, 'game_to_screen': 0.0, 'screen_to_game': 0.0}
textcnn: {'test_accuracy': 0.535827, 'test_macro_f1': 0.386317, 'majority_baseline': 0.597566, 'params': 196851}
mini_gpt: {'val_perplexity': 34.6845, 'unigram_baseline_perplexity': 2047.3361, 'params': 808128}
agreement: {'cnn_gpt_mean_cosine': 0.006765, 'probe_top3_jaccard_cnn_gpt